# SafeCut — context-aware profanity & slur censoring for sports highlights (English + Hinglish)

**Question I wanted to answer:** a word list alone either over-censors ("the *hooker* won the scrum", "Lund University") or leaks
real abuse that the speech recogniser misspells. How much does (1) fuzzy + Hinglish transliteration matching, (2) a RAG context check
against a written broadcast policy and (3) energy-based bleep-boundary refinement actually change the numbers?

**Why a synthetic benchmark:** there is no public video dataset with timestamped slurs. So I splice TTS words into clean MELD clips at
known sample positions → exact ground truth for *where* the word is, which is what bleep-boundary metrics need.

**Rules I followed:** no offensive word is written in the code (all come from the pinned LDNOOBW lists; printouts are masked like `f***`);
every threshold is tuned on a **dev split** and reported on a held-out **test split**; every number in the README is read from `results.json`
written by this notebook. Every cell ends with `✅` checks (assertions) so a silent failure stops the run.

### Kaggle settings (do these BEFORE Run All)
1. Settings → **Accelerator: GPU T4 x2** · **Internet: On**
2. Add-ons → Secrets → add **`GEMINI_API_KEY`** (attach it to this notebook)
3. Add Input → dataset **`zaber666/meld-dataset`** (MELD raw videos)
4. Add Input → Upload → new dataset **`safecut-assets`** containing `content_policy.md` and `innocent_sentences.jsonl` (from the `assets/` folder)

**If you run short of time:** set `SKIP_OCR=True` in Cell 1 (drops on-screen blurring, Cells 15–16), then skip the Streamlit tunnel (Cell 21).

In [ ]:
# Cell 1: install, machine check, paths, config
!pip install -q faster-whisper rapidfuzz easyocr faiss-cpu sentence-transformers google-genai edge-tts gTTS wordfreq indic-transliteration jiwer streamlit
import os, sys, glob, json, shutil, subprocess, time, random
import numpy as np, pandas as pd, torch

W = "/kaggle/working"; os.chdir(W); sys.path.insert(0, W); os.makedirs("safecut", exist_ok=True)
PROJ = f"{W}/safecut_run"
for d in ["lists", "tts", "clips", "innocent", "out", "cache"]:
    os.makedirs(f"{PROJ}/{d}", exist_ok=True)

CFG = dict(SEED=42, N_CLIPS=100, N_POS=50, N_OCR=20, N_DECOY=10, DEV_FRAC=0.3,
           WHISPER="large-v3", OCR_EVERY=0.5, SKIP_OCR=False)
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])

assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2"
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], "| torch", torch.__version__)
assert shutil.which("ffmpeg") and shutil.which("ffprobe"), "ffmpeg missing"
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])

def find_one(pattern):
    hits = glob.glob(pattern, recursive=True)
    return hits[0] if hits else None
POLICY = find_one("/kaggle/input/**/content_policy.md")
INNOCENT = find_one("/kaggle/input/**/innocent_sentences.jsonl")
MELD_DIR = find_one("/kaggle/input/**/MELD.Raw/test/output_repeated_splits_test")
print("policy:", POLICY, "\ninnocent:", INNOCENT, "\nMELD test videos:", MELD_DIR)
assert POLICY and INNOCENT, "Upload the safecut-assets dataset (Add Input -> Upload)"
assert MELD_DIR, "Add Input -> zaber666/meld-dataset"
print("MELD test mp4 files:", len(glob.glob(f"{MELD_DIR}/*.mp4")))

try:
    from kaggle_secrets import UserSecretsClient
    GEMINI_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
    print("Gemini key found, length:", len(GEMINI_KEY))
except Exception as e:
    GEMINI_KEY = None
    print("⚠️ No Gemini key -> the local Qwen fallback will be used for every LLM call:", str(e)[:80])
print("✅ Cell 1 OK")

⏱ **Runtime:** 3–5 min (pip install dominates; ~1 min on a re-run).

👉 **Manual step:** If an assert fails, fix the Kaggle setting it names (GPU, Internet, secret, datasets), then re-run this cell.

📋 **Check / paste back:** GPU line, MELD mp4 count, key length (never the key itself).

## Step 1 — Word lists and normalisation
The package code is written to `safecut/*.py` so the Streamlit app reuses exactly the code evaluated here.

In [ ]:
%%writefile safecut/__init__.py
"""SafeCut package."""

In [ ]:
%%writefile safecut/lexicon.py
"""Word lists, severity tiers, text normalisation and matching.
No offensive word is written in this file: every word comes from the downloaded lists."""
import hashlib, json, re, unicodedata, urllib.request
from dataclasses import dataclass, asdict
from pathlib import Path
from rapidfuzz import fuzz, process
from wordfreq import zipf_frequency

LDNOOBW_COMMIT = "5faf2ba42d7b1c0977169ec3611df25a3c08eb13"   # pinned so the lists never change under us
LDNOOBW_URL = "https://raw.githubusercontent.com/LDNOOBW/List-of-Dirty-Naughty-Obscene-and-Otherwise-Bad-Words/{commit}/{lang}"
LDNOOBW_SHA256 = {"en": "af851ecef1d5f212caba17339b12ac39cc2fef7d78c74876f67237644fcee8bd",
                  "hi": "83636e9b7a5260451e09e9e471f81dad9f082d107a82da77047e57d7856ed0cc"}
TIER_RANK = {"none": 0, "mild": 1, "strong": 2, "slur": 3}
DEVANAGARI = re.compile("[ऀ-ॿ]")


def download_lists(dest):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    paths = {}
    for lang in ("en", "hi"):
        p = dest / f"ldnoobw_{lang}.txt"
        if not p.exists():
            urllib.request.urlretrieve(LDNOOBW_URL.format(commit=LDNOOBW_COMMIT, lang=lang), p)
        digest = hashlib.sha256(p.read_bytes()).hexdigest()
        assert digest == LDNOOBW_SHA256[lang], f"{lang} list checksum changed: {digest}"
        paths[lang] = p
    return paths


@dataclass
class Entry:
    id: int
    text: str      # normalised, may contain spaces (phrases)
    lang: str      # 'en' or 'hi' (which list it came from)

    @property
    def tokens(self):
        return self.text.split()


def clean_entry(s):
    s = unicodedata.normalize("NFKC", s).strip().lower()
    return re.sub(r"\s+", " ", s)


def load_entries(paths):
    entries, seen = [], set()
    for lang, p in paths.items():
        for line in Path(p).read_text(encoding="utf-8").splitlines():
            t = clean_entry(line)
            if len(re.findall(r"[a-z]", t)) < 2 or t in seen:   # drop emoji / 1-letter entries / duplicates
                continue
            seen.add(t)
            entries.append(Entry(len(entries), t, lang))
    return entries


def mask(text):
    """Printable form of a listed word: first letter + stars. Used for every printout and the report."""
    return " ".join(w[0] + "*" * (len(w) - 1) for w in str(text).split())


def scrub(text, entries, repl="[term]"):
    """Remove every listed word (whole-word, any case) from free text, e.g. an LLM's reason."""
    for e in entries:
        if len(e.text) >= 3:
            text = re.sub(r"\b" + re.escape(e.text) + r"\b", repl, text, flags=re.I)
    return text


def norm_token(t):
    t = unicodedata.normalize("NFKC", str(t)).lower().replace("’", "'")
    t = re.sub(r"^[^a-z0-9*ऀ-ॿ]+|[^a-z0-9*ऀ-ॿ]+$", "", t)
    return t.replace("'", "")


def deva_to_roman(word):
    """Devanagari word -> loose Latin spelling (ITRANS, lower-cased, final schwa dropped)."""
    from indic_transliteration import sanscript
    r = sanscript.transliterate(word, sanscript.DEVANAGARI, sanscript.ITRANS)
    for nasal in (".N", ".n", "~N", "~n", "M"):
        r = r.replace(nasal, "n")
    r = re.sub(r"[^a-z]", "", r.lower())
    last = [c for c in word if DEVANAGARI.match(c)]
    if last and ("क" <= last[-1] <= "ह" or "क़" <= last[-1] <= "य़") and r.endswith("a"):
        r = r[:-1]          # Hindi drops the inherent final 'a' (कुत्ता keeps its matra, साल -> sal)
    return r


def skeleton(w):
    """Spelling-insensitive key for Hinglish: kh/k, ph/f, w/v, z/j, aa/a, ee/i, double letters, ..."""
    w = re.sub(r"[^a-z]", "", w.lower())
    w = w.replace("chh", "ch").replace("ch", "\x00").replace("ck", "k").replace("c", "k").replace("\x00", "c")
    for a, b in (("ph", "f"), ("kh", "k"), ("gh", "g"), ("th", "t"), ("dh", "d"), ("bh", "b"),
                 ("sh", "s"), ("jh", "j"), ("w", "v"), ("z", "j"), ("q", "k"),
                 ("aa", "a"), ("ee", "i"), ("ii", "i"), ("oo", "u"), ("uu", "u")):
        w = w.replace(a, b)
    w = re.sub(r"ey$", "e", w)
    w = re.sub(r"y$", "i", w)
    w = re.sub(r"(.)\1+", r"\1", w)
    return re.sub(r"(?<=[^aeiou])h$", "", w)


@dataclass
class Hit:
    i0: int            # first word index (inclusive)
    i1: int            # last word index (exclusive)
    start: float
    end: float
    heard: str         # normalised ASR/OCR text that matched (never printed unmasked)
    entry_id: int
    lang: str
    tier: str
    needs_context: bool
    match: str         # exact | phrase | translit | skeleton | fuzzy | masked
    score: float

    def to_dict(self):
        return asdict(self)


class Matcher:
    def __init__(self, entries, tiers, fuzzy_threshold=88, common_zipf=3.5, min_fuzzy_len=4):
        self.entries, self.tiers = entries, tiers
        self.fuzzy_threshold, self.common_zipf, self.min_fuzzy_len = fuzzy_threshold, common_zipf, min_fuzzy_len
        self.exact, self.phrases, self.skel = {}, {}, {}
        for e in entries:
            self.exact.setdefault(e.text.replace(" ", ""), e)
            if len(e.tokens) > 1:
                self.phrases.setdefault(tuple(e.tokens), e)
            sk = skeleton(e.text)
            if len(sk) >= 3:
                self.skel.setdefault(sk, e)
        self.fuzzy_keys = [k for k in self.exact if len(k) >= min_fuzzy_len]

    def info(self, e):
        t = self.tiers.get(e.id, {"tier": "strong", "needs_context": True})
        return t["tier"], bool(t["needs_context"])

    def match_token(self, tok):
        """Return (entry, match_type, score) or None for one normalised token."""
        if not tok:
            return None
        if "*" in tok:                                   # e.g. an ASR / caption that masks letters itself
            pat = re.compile("^" + re.escape(tok).replace(r"\*", "[a-z]") + "$")
            cands = [e for k, e in self.exact.items() if pat.match(k)]
            if cands and re.search(r"[a-z]", tok):
                e = max(cands, key=lambda e: TIER_RANK[self.info(e)[0]])
                return e, "masked", 100.0
            return None
        translit = bool(DEVANAGARI.search(tok))
        if translit:
            tok = deva_to_roman(tok)
            if not tok:
                return None
        if tok in self.exact:
            return self.exact[tok], ("translit" if translit else "exact"), 100.0
        if not translit and zipf_frequency(tok, "en") >= self.common_zipf:
            return None                                  # common English word: only exact matches allowed
        sk = skeleton(tok)
        if len(sk) >= 3 and sk in self.skel:
            return self.skel[sk], ("translit" if translit else "skeleton"), 100.0
        if self.fuzzy_threshold <= 100 and len(tok) >= self.min_fuzzy_len and self.fuzzy_keys:   # >100 = fuzzy off
            best = process.extractOne(tok, self.fuzzy_keys, scorer=fuzz.ratio, score_cutoff=self.fuzzy_threshold)
            if best:
                return self.exact[best[0]], ("translit" if translit else "fuzzy"), float(best[1])
        return None

    def find(self, words):
        """words: list of dicts with 'word', 'start', 'end' (Whisper output). Returns non-overlapping Hits."""
        toks = [norm_token(w["word"]) for w in words]
        used, hits = set(), []

        def add(i0, i1, e, kind, score):
            tier, ctx = self.info(e)
            hits.append(Hit(i0, i1, float(words[i0]["start"]), float(words[i1 - 1]["end"]),
                            " ".join(toks[i0:i1]), e.id, e.lang, tier, ctx, kind, score))
            used.update(range(i0, i1))

        for n in (4, 3, 2):                              # phrases and words split by the ASR
            for i in range(len(toks) - n + 1):
                span = toks[i:i + n]
                if used & set(range(i, i + n)) or not all(span):
                    continue
                if tuple(span) in self.phrases:
                    add(i, i + n, self.phrases[tuple(span)], "phrase", 100.0)
                elif "".join(span) in self.exact:
                    add(i, i + n, self.exact["".join(span)], "phrase", 100.0)
        for i, tok in enumerate(toks):
            if i not in used:
                m = self.match_token(tok)
                if m:
                    add(i, i + 1, *m)
        return sorted(hits, key=lambda h: h.start)

⏱ **Runtime:** < 1 s.

📋 **Check / paste back:** the printed output

In [ ]:
# Cell 3: load the pinned lists + unit tests for matching (tests generate misspellings from the list itself)
import importlib, safecut.lexicon as lx; importlib.reload(lx)
from safecut.lexicon import download_lists, load_entries, Matcher, norm_token, deva_to_roman, mask

ENTRIES = load_entries(download_lists(f"{PROJ}/lists"))
INN = [json.loads(l) for l in open(INNOCENT, encoding="utf-8")]
print("entries:", len(ENTRIES), {l: sum(e.lang == l for e in ENTRIES) for l in ("en", "hi")},
      "| multi-word phrases:", sum(len(e.tokens) > 1 for e in ENTRIES), "| innocent sentences:", len(INN))

M0 = Matcher(ENTRIES, {})
rng = random.Random(0)
single = [e for e in ENTRIES if len(e.tokens) == 1 and len(e.text) >= 6]
caught = 0
for e in rng.sample(single, 40):                       # drop one inner letter = a typical ASR misspelling
    i = rng.randrange(1, len(e.text) - 1)
    m = M0.match_token(e.text[:i] + e.text[i + 1:])
    caught += bool(m and m[0].id == e.id)
print(f"fuzzy test: {caught}/40 one-letter-deletion misspellings mapped back to the right entry")
assert caught >= 30

for w in ["class", "mall", "code", "assess", "scunthorpe", "analysis", "button", "cocktail"]:
    assert M0.match_token(w) is None, f"false hit on common word '{w}'"
print("common-word guard: 0 hits on 8 common English words that contain listed substrings")

e = single[0]
assert M0.match_token(e.text[0] + "*" * (len(e.text) - 1)) is not None
print("masked-spelling test passed")

assert all(r["target"] in {x.text for x in ENTRIES} for r in INN), "an innocent target is not in the lists"
txt_hit = sum(bool(M0.find([{"word": w, "start": 0, "end": 1} for w in r["text"].split()])) for r in INN)
hi_rows = [r for r in INN if r["lang"] == "hi"]
deva_hit = sum(any(M0.match_token(norm_token(w)) for w in r["tts_text"].split()) for r in hi_rows)
print(f"innocent sentences flagged by the word list on their text: {txt_hit}/{len(INN)} (should be all — that is the problem)")
print(f"Hinglish sentences flagged from Devanagari script via transliteration: {deva_hit}/{len(hi_rows)}")
assert txt_hit == len(INN)
print("✅ Cell 3 OK")

⏱ **Runtime:** < 10 s.

📋 **Check / paste back:** the counts and the two 'flagged' lines.

## Step 2 — Severity tiers (mild / strong / slur) with an LLM, cached
LDNOOBW has no severity, and it was built for *search filtering*, so it also contains clinical and ordinary words. I ask Gemini once
(batched, temperature 0) for `tier` + `needs_context` (does the string have innocent meanings?). The answer is cached in `tiers.json`.
`needs_context` is what later decides which hits go to the RAG check. If every Gemini model is rate-limited, the judge falls back to a local Qwen2.5-3B, and the cell prints which model answered.

In [ ]:
%%writefile safecut/llm.py
"""One JSON-in/JSON-out LLM judge: Gemini Flash models in order, then a local open model as fallback.
Every answer is cached on disk, so re-running the notebook never spends API quota twice."""
import hashlib, json, re, time
from collections import Counter
from pathlib import Path

GEMINI_CANDIDATES = ["models/gemini-flash-latest", "models/gemini-3.5-flash", "models/gemini-2.5-flash",
                     "models/gemini-flash-lite-latest", "models/gemini-2.5-flash-lite"]
SAFETY_CATEGORIES = ["HARM_CATEGORY_HARASSMENT", "HARM_CATEGORY_HATE_SPEECH",
                     "HARM_CATEGORY_SEXUALLY_EXPLICIT", "HARM_CATEGORY_DANGEROUS_CONTENT"]


def extract_json(text):
    if text is None:
        return None
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    for o, c in (("[", "]"), ("{", "}")):
        i, j = text.find(o), text.rfind(c)
        if i != -1 and j > i:
            try:
                return json.loads(text[i:j + 1])
            except Exception:
                continue
    return None


class Judge:
    def __init__(self, cache_file, api_key=None, candidates=GEMINI_CANDIDATES,
                 local_model="Qwen/Qwen2.5-3B-Instruct", local_device="cuda", log=print):
        self.cache_file = Path(cache_file)
        self.cache = json.loads(self.cache_file.read_text()) if self.cache_file.exists() else {}
        self.local_model_name, self.local_device, self.log = local_model, local_device, log
        self.usage, self.dead, self._local = Counter(), set(), None
        self.client, self.models = None, []
        if api_key:
            from google import genai
            from google.genai import types
            self.types = types
            self.client = genai.Client(api_key=api_key)
            try:
                listed = [(m.name, list(getattr(m, "supported_actions", None) or [])) for m in self.client.models.list()]
                by_short = {n.split("/")[-1]: n for n, _ in listed}
                self.models = [by_short[c.split("/")[-1]] for c in candidates if c.split("/")[-1] in by_short]
                if not self.models:      # names changed: use any text Flash model this key can call
                    self.models = [n for n, acts in listed if "flash" in n and (not acts or "generateContent" in acts)
                                   and not any(x in n for x in ("tts", "image", "live", "audio", "embed"))]
                self.log(f"Gemini key accepted: {len(listed)} models listed, {len(self.models)} usable Flash models")
            except Exception as e:
                self.log(f"model list failed ({str(e)[:120]}); trying candidates blindly")
                self.models = list(candidates)

    def _gemini_config(self):
        t = self.types
        return t.GenerateContentConfig(
            temperature=0.0, response_mime_type="application/json",
            safety_settings=[t.SafetySetting(category=c, threshold="BLOCK_NONE") for c in SAFETY_CATEGORIES])

    def _ask_gemini(self, prompt, validate):
        for m in self.models:
            if m in self.dead:
                continue
            for attempt in range(3):
                try:
                    resp = self.client.models.generate_content(model=m, contents=prompt, config=self._gemini_config())
                    obj = extract_json(resp.text)
                    if obj is not None and validate(obj):
                        return obj, m
                    self.log(f"  {m}: invalid/blocked answer (attempt {attempt + 1})")
                except Exception as e:
                    s = str(e)
                    delay = re.search(r"retry(?:Delay| in)[^0-9]*([0-9.]+)\s*s", s)
                    if ("429" in s or "RESOURCE_EXHAUSTED" in s) and delay and float(delay.group(1)) <= 65 and attempt < 2:
                        self.log(f"  {m}: per-minute limit, waiting {float(delay.group(1)):.0f}s")
                        time.sleep(float(delay.group(1)) + 1)
                        continue
                    if "503" in s or "UNAVAILABLE" in s or "500" in s:
                        self.log(f"  {m}: 503/500, retry in {5 * (attempt + 1)}s")
                        time.sleep(5 * (attempt + 1))
                        continue
                    self.log(f"  {m}: {s[:90]} -> next model")
                    self.dead.add(m)
                    break
        return None, None

    def _ask_local(self, prompt, validate):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        if self._local is None:
            self.log(f"loading local fallback {self.local_model_name} ...")
            tok = AutoTokenizer.from_pretrained(self.local_model_name)
            model = AutoModelForCausalLM.from_pretrained(
                self.local_model_name, dtype=torch.float16 if "cuda" in self.local_device else torch.float32
            ).to(self.local_device).eval()
            self._local = (tok, model)
        tok, model = self._local
        msgs = [{"role": "system", "content": "You are a careful broadcast standards editor. Reply with JSON only."},
                {"role": "user", "content": prompt}]
        text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)   # plain string in every version
        enc = tok(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=2048, do_sample=False, pad_token_id=tok.eos_token_id)
        obj = extract_json(tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True))
        return (obj, "local:" + self.local_model_name) if obj is not None and validate(obj) else (None, None)

    def ask(self, prompt, validate=lambda o: True):
        key = hashlib.sha1(prompt.encode()).hexdigest()
        if key in self.cache:
            self.usage["cache"] += 1
            return self.cache[key]["obj"], self.cache[key]["source"]
        obj, src = self._ask_gemini(prompt, validate) if self.client else (None, None)
        if obj is None and self.local_model_name:
            obj, src = self._ask_local(prompt, validate)
        if obj is None:
            raise RuntimeError("No LLM gave a valid answer for this prompt")
        self.usage[src] += 1
        self.cache[key] = {"obj": obj, "source": src}
        self.cache_file.write_text(json.dumps(self.cache))
        return obj, src


TIER_PROMPT = """You are building a broadcast-standards lexicon for live sports highlights in English and Hinglish
(romanised Hindi). For every numbered entry below decide:
- "tier": "mild" (mild swearing, crude or non-graphic sexual words, clinical sexual vocabulary),
          "strong" (strong swearing, graphic sexual terms, strong personal insults), or
          "slur" (slur or hate term aimed at a protected group: race, ethnicity, nationality, religion, caste,
                  gender, sexuality, disability).
- "needs_context": true if the entry also has a common innocent meaning (a name, place, animal, object,
  sports term, medical or legal word, or an ordinary word in another language); otherwise false.
Return ONLY a JSON list with one object per entry: [{{"i": <number>, "tier": "...", "needs_context": true/false}}, ...]

Entries (list language in brackets):
{items}"""


def tier_entries(judge, entries, batch=60, log=print):
    tiers, sources = {}, Counter()
    for b in range(0, len(entries), batch):
        chunk = entries[b:b + batch]
        items = "\n".join(f"{k}. [{e.lang}] {e.text}" for k, e in enumerate(chunk))
        want = set(range(len(chunk)))

        def ok(o):
            return (isinstance(o, list) and {d.get("i") for d in o if isinstance(d, dict)} >= want
                    and all(d.get("tier") in ("mild", "strong", "slur") for d in o if d.get("i") in want))
        obj, src = judge.ask(TIER_PROMPT.format(items=items), ok)
        sources[src] += 1
        for d in obj:
            if d.get("i") in want:
                tiers[chunk[d["i"]].id] = {"tier": d["tier"], "needs_context": bool(d.get("needs_context"))}
        log(f"  tiered {min(b + batch, len(entries))}/{len(entries)} via {src}")
    return tiers, sources

⏱ **Runtime:** < 1 s.

📋 **Check / paste back:** the printed output

In [ ]:
# Cell 5: tier every entry (≈9 Gemini calls the first time, 0 on re-runs)
import gc, safecut.llm as llm; importlib.reload(llm)
if "judge" in globals():                      # free a local model left on the GPU by an earlier failed run
    del judge; gc.collect(); torch.cuda.empty_cache()
print("Gemini key present:", bool(GEMINI_KEY), "(if False: Add-ons -> Secrets -> attach GEMINI_API_KEY, then re-run Cell 1)")
judge = llm.Judge(f"{PROJ}/cache/llm_cache.json", api_key=GEMINI_KEY,
                  local_device="cuda:1" if torch.cuda.device_count() > 1 else "cuda")
print("Gemini models available to try, in order:", judge.models)

TIERS_F = f"{PROJ}/tiers.json"
if os.path.exists(TIERS_F):
    TIERS = {int(k): v for k, v in json.load(open(TIERS_F)).items()}
    print("loaded cached tiers")
else:
    TIERS, src = llm.tier_entries(judge, ENTRIES)
    json.dump(TIERS, open(TIERS_F, "w"))
    print("tier sources:", dict(src))

assert len(TIERS) == len(ENTRIES) and all(t["tier"] in ("mild", "strong", "slur") for t in TIERS.values())
tdf = pd.DataFrame([{"lang": e.lang, **TIERS[e.id]} for e in ENTRIES])
print(pd.crosstab(tdf["lang"], tdf["tier"], margins=True))
print("share with innocent senses (needs_context):", round(tdf["needs_context"].mean(), 3))
r = random.Random(1)
for t in ("mild", "strong", "slur"):
    ex = [e for e in ENTRIES if TIERS[e.id]["tier"] == t]
    print(f"  {t:6s} examples (masked):", [mask(e.text) for e in r.sample(ex, min(4, len(ex)))])
tgt = {x.text: x.id for x in ENTRIES}
inn_ctx = sum(TIERS[tgt[x["target"]]]["needs_context"] for x in INN)
print(f"innocent-set targets the LLM marked needs_context: {inn_ctx}/{len(INN)}  (the rest can only be saved by the fuzzy/translit route)")
MATCHER = Matcher(ENTRIES, TIERS)
print("✅ Cell 5 OK | LLM usage:", dict(judge.usage))

⏱ **Runtime:** 30 s – 2 min with Gemini (each batch ~5–15 s; waits automatically on per-minute 429s). If Qwen fallback loads: +2–3 min.

👉 **Manual step:** Open `safecut_run/tiers.json` in the Output panel only if you want to audit tiers yourself; the notebook never prints the words.

📋 **Check / paste back:** the crosstab table, the needs_context share, and which model answered.

## Step 3 — Content policy → FAISS (RAG)
One chunk per rule (15 rules). Embeddings: `all-MiniLM-L6-v2`, cosine via inner product on normalised vectors.

In [ ]:
%%writefile safecut/rag.py
"""Content policy -> FAISS index -> top-k rules -> LLM context check for ambiguous hits."""
import re
import numpy as np


def load_policy(path):
    text = open(path, encoding="utf-8").read()
    rules = []
    for m in re.finditer(r"^## (R\d+)\s*[—-]+\s*(.+?)\n(.*?)(?=^## R\d+|\Z)", text, flags=re.M | re.S):
        rules.append({"rule_id": m.group(1), "title": m.group(2).strip(), "text": " ".join(m.group(3).split())})
    return rules


class PolicyIndex:
    def __init__(self, rules, model_name="sentence-transformers/all-MiniLM-L6-v2", device=None):
        import faiss
        from sentence_transformers import SentenceTransformer
        self.rules = rules
        self.model = SentenceTransformer(model_name, device=device)
        emb = self.model.encode([f"{r['title']}. {r['text']}" for r in rules], normalize_embeddings=True)
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(np.asarray(emb, dtype="float32"))

    def search(self, queries, k=3):
        q = self.model.encode(list(queries), normalize_embeddings=True)
        scores, idx = self.index.search(np.asarray(q, dtype="float32"), k)
        return [[dict(self.rules[j], score=float(s)) for s, j in zip(sr, ir)] for sr, ir in zip(scores, idx)]


def window_text(words, i0, i1, span=10):
    lo, hi = max(0, i0 - span), min(len(words), i1 + span)
    parts = [w["word"].strip() for w in words[lo:i0]] + ["⟦" + " ".join(w["word"].strip() for w in words[i0:i1]) + "⟧"] \
        + [w["word"].strip() for w in words[i1:hi]]
    return " ".join(parts)


CONTEXT_PROMPT = """You review speech-recognition transcripts from sports highlight clips for a general-audience broadcast.
In each item one span is marked with ⟦ ⟧. It matched the listed term shown (match type tells you how:
exact, phrase, translit = Hindi script converted to Latin, skeleton/fuzzy = approximate spelling, masked = letters hidden).
Decide, using ONLY the policy rules given with that item, whether the marked span is offensive in this context.
Return ONLY a JSON list: [{{"k": <item number>, "offensive": true/false, "severity": "none"|"mild"|"strong"|"slur",
"rule_id": "<one rule id from that item's rules>", "reason": "<max 20 words, do NOT repeat the word itself>"}}, ...]

{items}"""


def context_check(judge, index, items, batch=8, log=print):
    """items: dicts with keys key, window, term, match, tier, lang. Returns {key: decision}."""
    out = {}
    retrieved = index.search([it["window"].replace("⟦", "").replace("⟧", "") for it in items], k=3)
    for b in range(0, len(items), batch):
        chunk, rules = items[b:b + batch], retrieved[b:b + batch]
        blocks = []
        for k, (it, rs) in enumerate(zip(chunk, rules)):
            rtxt = "\n".join(f"   {r['rule_id']} ({r['title']}): {r['text']}" for r in rs)
            blocks.append(f"Item {k}: transcript: \"{it['window']}\"\n   listed term: \"{it['term']}\" | match: {it['match']}"
                          f" | lexicon tier: {it['tier']} | list language: {it['lang']}\n   Rules:\n{rtxt}")
        want = set(range(len(chunk)))

        def ok(o):
            return (isinstance(o, list) and {d.get("k") for d in o if isinstance(d, dict)} >= want
                    and all(isinstance(d.get("offensive"), bool) for d in o if d.get("k") in want))
        obj, src = judge.ask(CONTEXT_PROMPT.format(items="\n\n".join(blocks)), ok)
        for d in obj:
            if d.get("k") not in want:
                continue
            it, rs = chunk[d["k"]], rules[d["k"]]
            sev = d.get("severity") if d.get("severity") in ("none", "mild", "strong", "slur") else it["tier"]
            reason = str(d.get("reason", ""))
            for w in {it["term"], *it.get("heard", "").split()}:
                if len(w) >= 2:
                    reason = re.sub(re.escape(w), "[term]", reason, flags=re.I)
            out[it["key"]] = {"offensive": d["offensive"], "severity": sev if d["offensive"] else "none",
                              "rule_id": str(d.get("rule_id", "")), "reason": reason[:200], "source": src,
                              "retrieved": [r["rule_id"] for r in rs]}
        log(f"  context-checked {min(b + batch, len(items))}/{len(items)} via {src}")
    return out

⏱ **Runtime:** < 1 s.

📋 **Check / paste back:** the printed output

In [ ]:
# Cell 7: index the policy and sanity-check retrieval on 4 probe sentences I wrote (no listed words)
import safecut.rag as rag; importlib.reload(rag)
RULES = rag.load_policy(POLICY)
print("rules:", [r["rule_id"] for r in RULES])
assert len(RULES) == 15 and all(r["text"] for r in RULES)
INDEX = rag.PolicyIndex(RULES, device="cuda")
probes = [("The forward won the ball at the scrum, standard rugby position", "R05"),
          ("Fans held up a banner with rude writing on it", "R13"),
          ("He shouted abuse at the referee", "R04"),
          ("She studies sports science at a university in Sweden", "R06")]
ok = 0
for q, want in probes:
    got = [x["rule_id"] for x in INDEX.search([q], k=3)[0]]
    ok += want in got
    print(f"  {want} in top-3? {want in got} -> {got} | {q}")
assert ok >= 3, "retrieval is not finding obvious rules; check the policy file"
print(f"✅ Cell 7 OK (probe hit@3 = {ok}/4)")

⏱ **Runtime:** 20–40 s (MiniLM download).

👉 **Manual step:** Read the 15 rules in `content_policy.md` and edit them so the policy is yours; re-run from Cell 7 after edits.

📋 **Check / paste back:** the 4 probe lines.

## Step 4 — Synthetic benchmark with exact ground truth
* 100 clean MELD test clips (2.5–8 s, ≥5 words, gold transcript has **no** lexicon hit, max one clip per dialogue).
* 50 get one single-word list entry (≈70 % English list, 30 % Hindi list) spoken by edge-tts (gTTS fallback), inserted at a pause.
  The video freezes for exactly the inserted duration so audio and video stay in sync. The pause is filled with the clip's own room tone, not digital silence
  (silence would make boundary refinement unrealistically easy).
* 20 clips get offensive on-screen text (cv2) with exact boxes; 10 more get **clean** decoy text (to measure wrong blurs).
* 31 innocent sentences (assets file) are TTS'd: 24 English, 7 Hinglish read from Devanagari.
* Split: 30 % dev (tuning) / 70 % test (reporting), stratified by positive/negative.

In [ ]:
%%writefile safecut/audio.py
"""Audio I/O (via ffmpeg, never PyAV), energy curve and the three bleep-window strategies."""
import subprocess
import numpy as np

SR = 16000


def load_audio(path, sr=SR):
    out = subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-i", str(path), "-f", "f32le",
                          "-ac", "1", "-ar", str(sr), "-"], capture_output=True, check=True).stdout
    return np.frombuffer(out, np.float32).copy()


def save_wav(path, y, sr=SR):
    from scipy.io import wavfile
    wavfile.write(str(path), sr, (np.clip(y, -1, 1) * 32767).astype(np.int16))


def probe_duration(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(path)],
                       capture_output=True, text=True)
    try:
        return float(r.stdout.strip())
    except ValueError:
        return float("nan")


class EnergyCurve:
    """Short-time energy (dB) of a pre-emphasised signal. Pre-emphasis lifts weak fricatives like f/s/sh,
    which plain energy would treat as silence."""
    def __init__(self, y, sr=SR, win=0.02, hop=0.005, preemph=0.97):
        y = np.append(y[:1], y[1:] - preemph * y[:-1]) if len(y) > 1 else y
        n, h = int(win * sr), int(hop * sr)
        if len(y) < n:
            y = np.pad(y, (0, n - len(y)))
        frames = np.lib.stride_tricks.sliding_window_view(y, n)[::h]
        self.db = 20 * np.log10(np.sqrt((frames ** 2).mean(1)) + 1e-8)
        self.t = np.arange(len(self.db)) * hop + win / 2
        self.hop, self.dur = hop, len(y) / sr
        self.floor, self.peak = np.percentile(self.db, 10), np.percentile(self.db, 95)

    def idx(self, t):
        return int(np.clip(round((t - self.t[0]) / self.hop), 0, len(self.t) - 1))


def pad_window(s, e, pad, dur):
    return max(0.0, s - pad), min(dur, e + pad)


def refine_window(c, s, e, search=0.3, alpha=0.3, min_sil=0.03, guard=0.01):
    """Snap each edge to the nearest low-energy point within +-search seconds.
    Edge inside speech -> walk outward to the first quiet run of >= min_sil.
    Edge already in silence (Whisper too generous) -> walk inward to where speech starts."""
    low = c.db < c.floor + alpha * (c.peak - c.floor)
    run = max(1, int(round(min_sil / c.hop)))
    mid = c.idx((s + e) / 2)

    def quiet(j, step):
        k = [j + step * r for r in range(run)]
        return all(0 <= q < len(low) and low[q] for q in k)

    def snap(t, outward):                      # outward = -1 for start edge, +1 for end edge
        i, lim = c.idx(t), int(round(search / c.hop))
        if not low[i]:
            for d in range(lim + 1):
                j = i + outward * d
                if 0 <= j < len(low) and quiet(j, outward):
                    return c.t[j]
            lo, hi = sorted((i, i + outward * lim))
            lo, hi = max(lo, 0), min(hi, len(low) - 1)
            return c.t[lo + int(np.argmin(c.db[lo:hi + 1]))]
        for d in range(lim + 1):
            j = i - outward * d
            if (outward < 0 and j > mid) or (outward > 0 and j < mid) or not (0 <= j < len(low)):
                break
            if not low[j]:
                return c.t[j]
        return t

    return float(max(0.0, snap(s, -1) - guard)), float(min(c.dur, snap(e, +1) + guard))


def leak_overmute(gt, win):
    """leak = fraction of the true word NOT covered; over-mute = clean audio muted (ms)."""
    (gs, ge), (ws, we) = gt, win
    ov = max(0.0, min(ge, we) - max(gs, ws))
    return 1 - ov / (ge - gs), 1000 * ((we - ws) - ov)

In [ ]:
%%writefile safecut/metrics.py
import numpy as np


def wilson(k, n, z=1.96):
    """95% Wilson interval for a proportion k/n (works for small n and for 0 or n)."""
    if n == 0:
        return float("nan"), float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0, c - h), min(1, c + h)


def fmt_ci(k, n):
    p, lo, hi = wilson(k, n)
    return f"{100 * p:.1f}% [{100 * lo:.1f}, {100 * hi:.1f}] (n={n})"


def bootstrap_mean_ci(x, n_boot=10000, seed=0):
    x = np.asarray(x, float)
    if len(x) == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    means = x[rng.integers(0, len(x), (n_boot, len(x)))].mean(1)
    return x.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)


def box_coverage(gt, boxes):
    """Fraction of the gt box (x0,y0,x1,y1) covered by the union of boxes (pixel mask, exact)."""
    x0, y0, x1, y1 = [int(v) for v in gt]
    if x1 <= x0 or y1 <= y0:
        return 0.0
    m = np.zeros((y1 - y0, x1 - x0), bool)
    for bx0, by0, bx1, by1 in boxes:
        a0, b0 = max(int(bx0), x0) - x0, max(int(by0), y0) - y0
        a1, b1 = min(int(bx1), x1) - x0, min(int(by1), y1) - y0
        if a1 > a0 and b1 > b0:
            m[b0:b1, a0:a1] = True
    return float(m.mean())

In [ ]:
%%writefile safecut/synth.py
"""Synthetic benchmark builder: TTS a listed word, splice it into a clean clip at a pause,
freeze the video for the same time, and (optionally) draw text on frames. Ground truth is exact."""
import hashlib, subprocess
from pathlib import Path
import cv2
import numpy as np
from .audio import SR, EnergyCurve, load_audio

EN_VOICES = ["en-US-GuyNeural", "en-US-JennyNeural", "en-GB-RyanNeural", "en-GB-SoniaNeural",
             "en-IN-PrabhatNeural", "en-IN-NeerjaNeural"]
HINGLISH_VOICES = ["en-IN-PrabhatNeural", "en-IN-NeerjaNeural"]   # romanised Hindi said with an Indian accent
HI_VOICES = ["hi-IN-MadhurNeural", "hi-IN-SwaraNeural"]           # Devanagari sentences


def tts(text, voice, cache_dir):
    """edge-tts first, gTTS as fallback. Returns (16 kHz mono float32, engine name). Cached on disk."""
    cache_dir = Path(cache_dir); cache_dir.mkdir(parents=True, exist_ok=True)
    key = hashlib.sha1(f"{voice}|{text}".encode()).hexdigest()[:16]
    mp3, tag = cache_dir / f"{key}.mp3", cache_dir / f"{key}.engine"
    if not (mp3.exists() and tag.exists()):
        try:
            import edge_tts
            edge_tts.Communicate(text, voice).save_sync(str(mp3))
            engine = "edge-tts:" + voice
        except Exception:
            from gtts import gTTS
            lang = "hi" if voice.startswith("hi-") else "en"
            tld = {"en-GB": "co.uk", "en-IN": "co.in"}.get(voice[:5], "com")
            gTTS(text, lang=lang, tld=tld).save(str(mp3))
            engine = f"gtts:{lang}-{tld}"
        tag.write_text(engine)
    return load_audio(mp3), tag.read_text()


def trim_silence(y, rel_db=35):
    """Keep the part of a TTS clip that is within rel_db of its loudest 10 ms frame (= ground-truth word)."""
    n = SR // 100
    fr = y[: len(y) // n * n].reshape(-1, n)
    db = 20 * np.log10(np.sqrt((fr ** 2).mean(1)) + 1e-8)
    on = np.where(db > db.max() - rel_db)[0]
    return y[on[0] * n:(on[-1] + 1) * n].copy()


def room_tone(y, n, rng):
    """n samples of the clip's own quietest background (so pauses are not digital silence)."""
    w = SR // 10
    if len(y) < 2 * w:
        return np.zeros(n, np.float32)
    starts = np.arange(0, len(y) - w, w // 2)
    s = starts[np.argmin([np.sqrt((y[i:i + w] ** 2).mean()) for i in starts])]
    piece = y[s:s + w]
    reps = np.concatenate([np.roll(piece, int(rng.integers(w))) for _ in range(n // w + 1)])
    return reps[:n].astype(np.float32)


def speech_rms(y):
    n = SR // 50
    fr = y[: len(y) // n * n].reshape(-1, n)
    r = np.sqrt((fr ** 2).mean(1))
    return float(r[r >= np.median(r)].mean()) if len(r) else 0.05


def splice(y, word, fps, rng):
    """Insert `word` at a low-energy point between 15% and 85% of the clip, aligned to a video frame.
    Returns new audio, ground-truth (start, end) in seconds, the freeze frame index and number of frames."""
    word = word * (speech_rms(y) / (speech_rms(word) + 1e-8)) * 10 ** (rng.uniform(-3, 3) / 20)
    c = EnergyCurve(y)
    dur = len(y) / SR
    t = c.t[(c.t > 0.15 * dur) & (c.t < 0.85 * dur)]
    d = c.db[(c.t > 0.15 * dur) & (c.t < 0.85 * dur)]
    quiet = t[d < c.floor + 0.25 * (c.peak - c.floor)]
    t_ins = float(rng.choice(quiet)) if len(quiet) else float(t[np.argmin(d)])
    f = int(round(t_ins * fps))
    p = int(round(f / fps * SR))
    g1 = int(rng.uniform(0.04, 0.20) * SR)
    n_freeze = int(np.ceil((g1 + len(word) + 0.04 * SR) / SR * fps))
    total = int(round(n_freeze / fps * SR))
    seg = room_tone(y, total, rng)
    seg[g1:g1 + len(word)] += word
    y2 = np.concatenate([y[:p], seg, y[p:]]).astype(np.float32)
    gt = ((p + g1) / SR, (p + g1 + len(word)) / SR)
    return y2, gt, f, n_freeze


def draw_text(frame, text, x, y, scale, thick, color):
    """Caption-style text on a dark banner. Returns the exact text box (x0, y0, x1, y1)."""
    font = cv2.FONT_HERSHEY_DUPLEX
    (w, h), base = cv2.getTextSize(text, font, scale, thick)
    over = frame.copy()
    cv2.rectangle(over, (x - 8, y - h - 8), (x + w + 8, y + base + 8), (20, 20, 20), -1)
    cv2.addWeighted(over, 0.6, frame, 0.4, 0, frame)
    cv2.putText(frame, text, (x, y), font, scale, color, thick, cv2.LINE_AA)
    return (x, y - h, x + w, y + base)


def place_text(text, W, H, rng):
    scale = float(rng.uniform(1.0, 2.0)); thick = int(rng.integers(2, 4))
    (w, h), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_DUPLEX, scale, thick)
    x = int(rng.integers(20, max(21, W - w - 20)))
    y = int(rng.integers(h + 20, max(h + 21, H - base - 20)))
    color = [(255, 255, 255), (0, 255, 255), (255, 255, 0)][int(rng.integers(3))]
    return dict(x=x, y=y, scale=scale, thick=thick, color=color)


def build_video(src, out_noaudio, freeze_f, n_freeze, overlays):
    """Copy src frames, repeat frame `freeze_f` n_freeze times, draw overlays (times in the OUTPUT timeline).
    Returns fps, (W, H), n_frames, per-frame list of ground-truth boxes for offensive overlays."""
    cap = cv2.VideoCapture(str(src))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frames = []
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        frames.append(fr)
    cap.release()
    if not frames:
        raise RuntimeError(f"no frames in {src}")
    H, W = frames[0].shape[:2]
    if freeze_f is not None and n_freeze:
        f = min(freeze_f, len(frames) - 1)
        frames = frames[:f] + [frames[f]] * n_freeze + frames[f:]
    vw = cv2.VideoWriter(str(out_noaudio), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    gt_boxes = [[] for _ in frames]
    for k, fr in enumerate(frames):
        fr = fr.copy()
        t = k / fps
        for ov in overlays:
            if ov["t0"] <= t < ov["t1"]:
                box = draw_text(fr, ov["text"], ov["x"], ov["y"], ov["scale"], ov["thick"], ov["color"])
                ov["box"] = box
                if ov["offensive"]:
                    gt_boxes[k].append(box)
        vw.write(fr)
    vw.release()
    return fps, (W, H), len(frames), gt_boxes


def mux(video_noaudio, wav, out_mp4):
    subprocess.run(["ffmpeg", "-nostdin", "-y", "-loglevel", "error", "-i", str(video_noaudio), "-i", str(wav),
                    "-map", "0:v", "-map", "1:a", "-c:v", "libx264", "-preset", "veryfast", "-crf", "23",
                    "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "128k", "-shortest", str(out_mp4)], check=True)

In [ ]:
%%writefile safecut/render.py
"""ffmpeg rendering: 1 kHz bleep over mild/strong words, drop slur segments, concat into a reel."""
import subprocess
import numpy as np
from .audio import probe_duration


def _expr(wins):
    return "+".join(f"between(t,{s:.3f},{e:.3f})" for s, e in wins) or "0"


def render_clip(video_in, audio_in, out, bleeps=(), drops=(), tone_gain=0.3):
    """video_in: video stream source; audio_in: audio source (wav or same mp4). Times in input seconds."""
    dur = probe_duration(audio_in)
    B, D = _expr(bleeps), _expr(drops)
    fc = (f"[1:a]aresample=48000,aformat=channel_layouts=mono,asetnsamples=n=64:p=0,"
          f"volume=volume='if(gt({B},0),0,1)':eval=frame[sp];"
          f"sine=frequency=1000:sample_rate=48000:duration={dur:.3f},asetnsamples=n=64:p=0,"
          f"volume=volume='if(gt({B},0),{tone_gain},0)':eval=frame[tn];"
          f"[sp][tn]amix=inputs=2:duration=first:normalize=0[mx];"
          f"[0:v]select='not(gt({D},0))',setpts=N/FRAME_RATE/TB[v];"
          f"[mx]aselect='not(gt({D},0))',asetpts=N/SR/TB[a]")
    cmd = ["ffmpeg", "-nostdin", "-y", "-loglevel", "error", "-i", str(video_in), "-i", str(audio_in),
           "-filter_complex", fc, "-map", "[v]", "-map", "[a]", "-c:v", "libx264", "-preset", "veryfast",
           "-crf", "23", "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "128k", "-movflags", "+faststart", str(out)]
    subprocess.run(cmd, check=True)
    return cmd


def concat(clips, out, W=1280, H=720, fps=24):
    ins, fc = [], ""
    for i, c in enumerate(clips):
        ins += ["-i", str(c)]
        fc += (f"[{i}:v]scale={W}:{H}:force_original_aspect_ratio=decrease,pad={W}:{H}:(ow-iw)/2:(oh-ih)/2,"
               f"fps={fps},setsar=1[v{i}];[{i}:a]aresample=48000,aformat=channel_layouts=mono[a{i}];")
    fc += "".join(f"[v{i}][a{i}]" for i in range(len(clips))) + f"concat=n={len(clips)}:v=1:a=1[v][a]"
    subprocess.run(["ffmpeg", "-nostdin", "-y", "-loglevel", "error", *ins, "-filter_complex", fc, "-map", "[v]",
                    "-map", "[a]", "-c:v", "libx264", "-preset", "veryfast", "-crf", "23", "-pix_fmt", "yuv420p",
                    "-c:a", "aac", "-b:a", "128k", str(out)], check=True)


def tone_ratio(y, sr, f=1000.0, bw=30.0):
    """Share of signal power within +-bw Hz of f (1.0 = pure tone)."""
    if len(y) < 32:
        return 0.0
    spec = np.abs(np.fft.rfft(y * np.hanning(len(y)))) ** 2
    freqs = np.fft.rfftfreq(len(y), 1 / sr)
    return float(spec[np.abs(freqs - f) <= bw].sum() / (spec.sum() + 1e-12))

⏱ **Runtime:** < 1 s.

📋 **Check / paste back:** the printed output

In [ ]:
# Cell 10: build the benchmark (resumable: finished clips are skipped)
import cv2, safecut.synth as sy, safecut.audio as au; importlib.reload(au); importlib.reload(sy)
from safecut.audio import SR, load_audio, save_wav, probe_duration
BENCH_F = f"{PROJ}/bench.json"
rng = np.random.default_rng(CFG["SEED"])

df = pd.read_csv("https://raw.githubusercontent.com/declare-lab/MELD/master/data/MELD/test_sent_emo.csv",
                 encoding="utf-8", encoding_errors="replace")
df["clip"] = "dia" + df["Dialogue_ID"].astype(str) + "_utt" + df["Utterance_ID"].astype(str)
df["path"] = MELD_DIR + "/" + df["clip"] + ".mp4"
df = df[df["path"].map(os.path.exists)].copy()
df["n_words"] = df["Utterance"].astype(str).str.split().str.len()
df["clean"] = df["Utterance"].astype(str).map(
    lambda t: not MATCHER.find([{"word": w, "start": 0, "end": 0} for w in t.split()]))
print("MELD test rows with video:", len(df), "| clean gold text:", int(df["clean"].sum()))
cand = df[(df["n_words"] >= 5) & df["clean"]].sample(frac=1, random_state=CFG["SEED"]).drop_duplicates("Dialogue_ID")

chosen = []
for r in cand.itertuples():
    d = probe_duration(r.path)
    if 2.5 <= d <= 8.0 and len(load_audio(r.path)) > SR:
        chosen.append(r)
    if len(chosen) == CFG["N_CLIPS"]:
        break
assert len(chosen) == CFG["N_CLIPS"], f"only {len(chosen)} usable clips"

pool = [e for e in ENTRIES if len(e.tokens) == 1 and len(e.text) >= 3]
en_pool, hi_pool = [e for e in pool if e.lang == "en"], [e for e in pool if e.lang == "hi"]
n_hi = round(0.3 * CFG["N_POS"])
ins_words = list(rng.choice(en_pool, CFG["N_POS"] - n_hi, replace=False)) + list(rng.choice(hi_pool, n_hi, replace=False))
rng.shuffle(ins_words)
ocr_ids = set(rng.choice(CFG["N_CLIPS"], CFG["N_OCR"] + CFG["N_DECOY"], replace=False).tolist())
ocr_list = sorted(ocr_ids); decoy_ids = set(ocr_list[CFG["N_OCR"]:]); ocr_ids = set(ocr_list[:CFG["N_OCR"]])
ocr_words = [e for e in pool if len(e.text) >= 4]
DECOYS = ["GOAL!", "REPLAY", "FULL TIME", "SIX!", "WICKET", "WHAT A SAVE", "4-2", "LIVE"]
pos_flag = np.array([1] * CFG["N_POS"] + [0] * (CFG["N_CLIPS"] - CFG["N_POS"]))
dev = np.zeros(CFG["N_CLIPS"], bool)
for v in (0, 1):
    idx = np.where(pos_flag == v)[0]
    dev[rng.choice(idx, int(round(CFG["DEV_FRAC"] * len(idx))), replace=False)] = True

BENCH = json.load(open(BENCH_F)) if os.path.exists(BENCH_F) else {}
t0 = time.time()
for i, r in enumerate(chosen):
    cid = f"c{i:03d}_{r.clip}"
    if cid in BENCH and os.path.exists(f"{PROJ}/clips/{cid}.mp4"):
        continue
    y = load_audio(r.path)
    fps = cv2.VideoCapture(r.path).get(cv2.CAP_PROP_FPS) or 25.0
    meta = dict(clip=cid, src=r.path, gold=str(r.Utterance), split="dev" if dev[i] else "test",
                pos=bool(pos_flag[i]), fps=fps, overlays=[])
    freeze_f = n_freeze = None
    if pos_flag[i]:
        e = ins_words[i]
        voices = sy.HINGLISH_VOICES if e.lang == "hi" else sy.EN_VOICES
        w, engine = sy.tts(e.text, voices[int(rng.integers(len(voices)))], f"{PROJ}/tts")
        w = sy.trim_silence(w)
        y, gt, freeze_f, n_freeze = sy.splice(y, w, fps, rng)
        meta.update(entry_id=e.id, lang=e.lang, tier=TIERS[e.id]["tier"], gt=gt, tts=engine)
    dur = len(y) / SR
    H, Wd = int(cv2.VideoCapture(r.path).get(4)), int(cv2.VideoCapture(r.path).get(3))
    if i in ocr_ids:
        e = ocr_words[int(rng.integers(len(ocr_words)))]
        text = [e.text.upper(), e.text.title(), e.text][int(rng.integers(3))]
        t_a = float(rng.uniform(0.2, 0.5 * dur)); t_b = min(dur - 0.2, t_a + float(rng.uniform(1.0, 2.5)))
        meta["overlays"].append(dict(t0=t_a, t1=t_b, text=text, offensive=True, entry_id=e.id, **sy.place_text(text, Wd, H, rng)))
    if i in decoy_ids or (i in ocr_ids and rng.random() < 0.5):
        text = DECOYS[int(rng.integers(len(DECOYS)))]
        t_a = float(rng.uniform(0.1, 0.6 * dur))
        meta["overlays"].append(dict(t0=t_a, t1=min(dur - 0.1, t_a + 1.5), text=text, offensive=False, **sy.place_text(text, Wd, H, rng)))
    noa, wav, mp4 = f"{PROJ}/clips/{cid}_noaudio.mp4", f"{PROJ}/clips/{cid}.wav", f"{PROJ}/clips/{cid}.mp4"
    fps2, size, n_frames, _ = sy.build_video(r.path, noa, freeze_f, n_freeze, meta["overlays"])
    save_wav(wav, y); sy.mux(noa, wav, mp4); os.remove(noa)
    for ov in meta["overlays"]:
        ov["color"] = list(ov["color"]); ov["text"] = "<listed>" if ov["offensive"] else ov["text"]   # never store the word in text form
    meta.update(dur=dur, n_frames=n_frames, size=size, mp4_dur=probe_duration(mp4))
    BENCH[cid] = meta
    if i % 10 == 0:
        json.dump(BENCH, open(BENCH_F, "w"))
json.dump(BENCH, open(BENCH_F, "w"))
print(f"built in {time.time() - t0:.0f}s")

B = pd.DataFrame(BENCH.values())
assert len(B) == CFG["N_CLIPS"] and B["pos"].sum() == CFG["N_POS"]
for m in BENCH.values():
    assert abs(m["mp4_dur"] - m["dur"]) < 0.15, (m["clip"], m["mp4_dur"], m["dur"])
    if m["pos"]:
        assert 0 < m["gt"][0] < m["gt"][1] < m["dur"]
print(pd.crosstab(B["split"], B["pos"], margins=True))
P = B[B["pos"]]
print("inserted words by list:", P["lang"].value_counts().to_dict(), "| tier:", P["tier"].value_counts().to_dict())
print("TTS engines:", P["tts"].str.split(":").str[0].value_counts().to_dict())
print("true word duration (s): mean", round((P["gt"].str[1] - P["gt"].str[0]).mean(), 3))
print("clips with offensive overlay:", sum(any(o["offensive"] for o in m["overlays"]) for m in BENCH.values()),
      "| clips with clean decoy text:", sum(any(not o["offensive"] for o in m["overlays"]) for m in BENCH.values()))
print("✅ Cell 10 OK")

⏱ **Runtime:** 4–8 min first time (ffprobe on candidates, ~50 TTS calls, writing 100 videos). Resumable — re-run continues where it stopped.

📋 **Check / paste back:** the split table, words by list/tier, TTS engine counts and overlay counts.

In [ ]:
# Cell 11: TTS the 31 innocent sentences (English voices for English, Hindi voices reading Devanagari)
INN_META = {}
for k, r in enumerate(INN):
    voices = sy.HI_VOICES if r["lang"] == "hi" else sy.EN_VOICES
    y, engine = sy.tts(r["tts_text"], voices[k % len(voices)], f"{PROJ}/tts")
    wav = f"{PROJ}/innocent/{r['id']}.wav"; save_wav(wav, y)
    INN_META[r["id"]] = dict(clip=r["id"], lang=r["lang"], category=r["category"], target_id=tgt[r["target"]],
                             wav=wav, dur=len(y) / SR, tts=engine)
assert len(INN_META) == len(INN) and all(m["dur"] > 1.0 for m in INN_META.values())
print(pd.DataFrame(INN_META.values()).groupby("lang")["dur"].describe()[["count", "mean"]])
print("✅ Cell 11 OK")

⏱ **Runtime:** 30–60 s.

📋 **Check / paste back:** the printed output

## Step 5 — ASR: faster-whisper large-v3, word timestamps, language auto-detect
Audio is decoded with ffmpeg into numpy (passing file paths broke on Kaggle's PyAV in my previous project). Auto-detect matters: Hinglish speech comes out in Devanagari and must be transliterated (Step 6).

In [ ]:
%%writefile safecut/asr.py
def transcribe(model, y, language=None, beam_size=5):
    """faster-whisper on a 16 kHz float array (arrays avoid the PyAV decoding bug seen on Kaggle)."""
    segs, info = model.transcribe(y, language=language, beam_size=beam_size, word_timestamps=True,
                                  condition_on_previous_text=False, vad_filter=False)
    words = [{"word": w.word, "start": float(w.start), "end": float(w.end), "prob": float(w.probability),
              "seg_start": float(s.start), "seg_end": float(s.end)}
             for s in segs for w in (s.words or [])]
    return words, info.language, float(info.language_probability)

⏱ **Runtime:** < 1 s.

📋 **Check / paste back:** the printed output

In [ ]:
# Cell 13: transcribe everything (cached), check timestamps, and measure WER on the clean clips vs MELD gold text
import jiwer, re, safecut.asr as asr_mod; importlib.reload(asr_mod)
from faster_whisper import WhisperModel
whisper = WhisperModel(CFG["WHISPER"], device="cuda", compute_type="float16")
tone = (0.1 * np.sin(2 * np.pi * 440 * np.arange(SR) / SR)).astype(np.float32)
list(whisper.transcribe(tone)[0]); print("Whisper GPU smoke test OK")

ASR_F = f"{PROJ}/asr.json"
ASR = json.load(open(ASR_F)) if os.path.exists(ASR_F) else {}
jobs = [(c, f"{PROJ}/clips/{c}.wav") for c in BENCH] + [(c, m["wav"]) for c, m in INN_META.items()]
asr_time, asr_audio = 0.0, 0.0
for c, wav in jobs:
    if c in ASR:
        continue
    y = load_audio(wav); t = time.time()
    words, lang, lp = asr_mod.transcribe(whisper, y)
    ASR[c] = dict(words=words, lang=lang, lang_p=lp, sec=time.time() - t, dur=len(y) / SR)
json.dump(ASR, open(ASR_F, "w"))
tot_sec = sum(v["sec"] for v in ASR.values()); tot_dur = sum(v["dur"] for v in ASR.values())
print(f"ASR: {tot_dur:.0f}s of audio in {tot_sec:.0f}s -> {tot_dur / tot_sec:.1f}x real-time")

bad = [c for c, v in ASR.items() for a, b in zip(v["words"], v["words"][1:]) if b["start"] + 1e-3 < a["start"]]
assert not bad, f"non-monotonic timestamps in {bad[:3]}"
assert all(w["end"] <= v["dur"] + 0.5 for v in ASR.values() for w in v["words"])
empty = [c for c, v in ASR.items() if not v["words"]]
print("clips with no words:", len(empty)); assert len(empty) <= 3
norm = lambda s: re.sub(r"[^a-z0-9' ]", " ", s.lower()).split()
neg = [m for m in BENCH.values() if not m["pos"]]
wer = jiwer.wer([" ".join(norm(m["gold"])) for m in neg], [" ".join(norm(" ".join(w["word"] for w in ASR[m["clip"]]["words"]))) for m in neg])
print(f"WER on {len(neg)} untouched MELD clips vs gold transcript: {wer:.3f}")
print("detected language, innocent Hinglish clips:", {c: ASR[c]["lang"] for c, m in INN_META.items() if m["lang"] == "hi"})
print("✅ Cell 13 OK")

⏱ **Runtime:** 3–6 min (large-v3 download ~1.5 min + ~131 clips). Cached on re-run.

📋 **Check / paste back:** the ×real-time line, WER line and the detected languages.

## Step 6 — Detection: exact + phrase + fuzzy (rapidfuzz) + Hinglish skeleton/transliteration
Two knobs (fuzzy threshold, common-word guard) are tuned on **dev only** by F1. A hit is a true positive if it overlaps the true word interval (±0.1 s).

In [ ]:
# Cell 14: tune on dev, evaluate detection on test, error analysis of misses
from safecut.metrics import fmt_ci, wilson
TOL = 0.1
def score(matcher, clips):
    tp = fp = fn = 0; rows = []
    for c in clips:
        m = BENCH[c]; hits = matcher.find(ASR[c]["words"])
        got = False
        for h in hits:
            if m["pos"] and h.end >= m["gt"][0] - TOL and h.start <= m["gt"][1] + TOL:
                got = True
            else:
                fp += 1
        if m["pos"]:
            tp += got; fn += not got
        rows.append(dict(clip=c, pos=m["pos"], found=got, n_hits=len(hits)))
    return tp, fp, fn, rows
f1 = lambda tp, fp, fn: 2 * tp / max(1, 2 * tp + fp + fn)
DEV = [c for c, m in BENCH.items() if m["split"] == "dev"]; TEST = [c for c, m in BENCH.items() if m["split"] == "test"]
grid = []
for thr in (80, 85, 88, 92, 95, 101):          # 101 = fuzzy matching off
    for z in (3.0, 3.5, 4.0):
        tp, fp, fn, _ = score(lx.Matcher(ENTRIES, TIERS, fuzzy_threshold=thr, common_zipf=z), DEV)
        grid.append(dict(thr=thr, zipf=z, tp=tp, fp=fp, fn=fn, f1=f1(tp, fp, fn)))
G = pd.DataFrame(grid).sort_values(["f1", "fp"], ascending=[False, True]); print(G.head(6).to_string(index=False))
best = G.iloc[0]; FUZZY_THR, ZIPF = int(best.thr), float(best.zipf)
MATCHER = lx.Matcher(ENTRIES, TIERS, fuzzy_threshold=FUZZY_THR, common_zipf=ZIPF)
print(f"chosen on dev: fuzzy_threshold={FUZZY_THR}, common_zipf={ZIPF}")

tp, fp, fn, rows = score(MATCHER, TEST)
print(f"TEST detection precision: {fmt_ci(tp, tp + fp)}\nTEST detection recall:    {fmt_ci(tp, tp + fn)}")
tp0, fp0, fn0, _ = score(lx.Matcher(ENTRIES, TIERS, fuzzy_threshold=101, common_zipf=0), TEST)   # zipf guard 0 = every token "common" -> exact/phrase only
print(f"exact/phrase-only baseline on TEST: precision {fmt_ci(tp0, tp0 + fp0)} | recall {fmt_ci(tp0, tp0 + fn0)}")

R = pd.DataFrame(rows).merge(B[["clip", "lang", "tier"]], on="clip")
print("TEST recall by list language:", R[R.pos].groupby("lang")["found"].agg(["sum", "count"]).to_dict("index"))
exact_heard = [any(lx.norm_token(w["word"]) == ENTRIES[BENCH[c]["entry_id"]].text for w in ASR[c]["words"])
               for c in BENCH if BENCH[c]["pos"]]
print(f"inserted words that Whisper spelled exactly like the list: {sum(exact_heard)}/{len(exact_heard)}")
print("\nmissed TEST words — what Whisper heard nearest the true time (masked):")
for c in R.loc[R["pos"] & ~R["found"], "clip"]:
    m = BENCH[c]; ws = [w for w in ASR[c]["words"] if w["end"] >= m["gt"][0] - 0.2 and w["start"] <= m["gt"][1] + 0.2]
    print(f"  {c[:5]} [{m['lang']}] target {mask(ENTRIES[m['entry_id']].text):12s} heard: {[mask(lx.norm_token(w['word'])) for w in ws]}")
HITS = {c: MATCHER.find(v["words"]) for c, v in ASR.items()}
print("match types over all clips:", pd.Series([h.match for hs in HITS.values() for h in hs]).value_counts().to_dict())
print("✅ Cell 14 OK")

⏱ **Runtime:** 10–30 s.

📋 **Check / paste back:** the dev grid head, TEST precision/recall with CIs, the baseline line, by-language recall and the miss list.

## Step 7 — RAG context check on ambiguous hits only
Routed to the LLM when `needs_context` is true **or** the match was not exact. Top-3 rules retrieved per hit; batched 8 hits per call; reason text is scrubbed of the word.

In [ ]:
%%writefile safecut/pipeline.py
"""Glue used by both the notebook and the Streamlit app: hits -> decisions -> actions -> rendered clip + report."""
import json, time
from pathlib import Path
from .asr import transcribe
from .audio import SR, EnergyCurve, load_audio, refine_window, save_wav
from .lexicon import mask, scrub
from .rag import context_check, window_text
from .render import render_clip

DEFAULT_RULE = {"mild": "R01", "strong": "R02", "slur": "R03"}
ACTION = {"mild": "bleep", "strong": "bleep", "slur": "drop", "none": "keep"}


def needs_check(hit):
    """Ambiguous = the lexicon says the word has innocent senses, or the match was not exact."""
    return hit.needs_context or hit.match not in ("exact", "phrase")


def context_items(clip_id, words, hits, entries):
    return [{"key": f"{clip_id}#{k}", "window": window_text(words, h.i0, h.i1), "term": entries[h.entry_id].text,
             "heard": h.heard, "match": h.match, "tier": h.tier, "lang": h.lang}
            for k, h in enumerate(hits) if needs_check(h)]


def decide(hit, decision, use_context=True):
    if use_context and needs_check(hit) and decision is not None:
        sev = decision["severity"] if decision["offensive"] else "none"
        return {"severity": sev, "rule_id": decision["rule_id"], "reason": decision["reason"], "judge": decision["source"]}
    return {"severity": hit.tier, "rule_id": DEFAULT_RULE[hit.tier], "reason": "unambiguous lexicon match",
            "judge": "lexicon"}


def plan_actions(clip_id, y, words, hits, decisions, cfg, use_context=True, entries=()):
    """Returns report events + bleep and drop windows (seconds)."""
    curve = EnergyCurve(y)
    events, bleeps, drops = [], [], []
    for k, h in enumerate(hits):
        d = decide(h, decisions.get(f"{clip_id}#{k}"), use_context)
        win = refine_window(curve, h.start, h.end, **cfg["refine"])
        act = ACTION[d["severity"]]
        if act == "bleep":
            bleeps.append(win)
        elif act == "drop":
            w = words[h.i0]
            win = (min(win[0], w.get("seg_start", win[0])), max(win[1], w.get("seg_end", win[1])))
            drops.append(win)
        events.append({"clip": clip_id, "modality": "audio", "start": round(win[0], 3), "end": round(win[1], 3),
                       "tier": d["severity"], "lexicon_tier": h.tier, "match": h.match, "action": act,
                       "rule_id": d["rule_id"], "reason": scrub(d["reason"], entries), "judge": d["judge"], "heard_masked": mask(h.heard)})
    return events, bleeps, drops


def process_video(path, ctx, workdir):
    """Full pipeline on one uploaded video. ctx: dict with whisper, matcher, entries, judge, index, reader, cfg."""
    from .ocr import blur_video, ocr_hits, read_frames, sample_indices
    workdir = Path(workdir); workdir.mkdir(parents=True, exist_ok=True)
    T, cfg, clip_id = {}, ctx["cfg"], Path(path).stem
    t = time.time(); y = load_audio(path); wav = workdir / f"{clip_id}.wav"; save_wav(wav, y); T["decode"] = time.time() - t
    t = time.time(); words, lang, _ = transcribe(ctx["whisper"], y); T["asr"] = time.time() - t
    t = time.time(); hits = ctx["matcher"].find(words); T["detect"] = time.time() - t
    t = time.time()
    items = context_items(clip_id, words, hits, ctx["entries"])
    decisions = context_check(ctx["judge"], ctx["index"], items, log=lambda *a: None) if items else {}
    T["context"] = time.time() - t
    events, bleeps, drops = plan_actions(clip_id, y, words, hits, decisions, cfg, entries=ctx["entries"])
    t = time.time()
    frames, fps = read_frames(path)
    si = sample_indices(len(frames), fps, cfg["ocr_every"])
    sb, vis = [], []
    for k in si:
        hs = ocr_hits(ctx["reader"], ctx["matcher"], frames[k])
        sb.append([h["box"] for h in hs])
        for h in hs:
            vis.append({"clip": clip_id, "modality": "video", "start": round(k / fps, 3),
                        "end": round(k / fps + cfg["ocr_every"], 3), "tier": h["tier"], "action": "blur",
                        "rule_id": "R13", "reason": "listed word visible on screen", "judge": "lexicon"})
    blurred = workdir / f"{clip_id}_blur.mp4"
    blur_video(frames, fps, si, sb, blurred, mode=cfg["ocr_mode"])
    T["ocr_blur"] = time.time() - t
    t = time.time(); out = workdir / f"{clip_id}_safe.mp4"; render_clip(blurred, wav, out, bleeps, drops); T["render"] = time.time() - t
    report = {"clip": clip_id, "duration_s": round(len(y) / SR, 3), "asr_language": lang,
              "events": events + vis, "timings_s": {k: round(v, 3) for k, v in T.items()}}
    (workdir / f"{clip_id}_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))
    return report, out

⏱ **Runtime:** < 1 s.

📋 **Check / paste back:** the printed output

In [ ]:
# Cell 16: context-check every routed hit (bench + innocent), then measure what the check buys and what it costs
import safecut.pipeline as pl; importlib.reload(pl)
items = []
for c, hs in HITS.items():
    items += pl.context_items(c, ASR[c]["words"], hs, ENTRIES)
print("hits total:", sum(map(len, HITS.values())), "| routed to context check:", len(items))
t = time.time(); DECISIONS = rag.context_check(judge, INDEX, items) if items else {}; CTX_SEC = time.time() - t
assert set(DECISIONS) == {it["key"] for it in items}
cited_ok = np.mean([d["rule_id"] in d["retrieved"] for d in DECISIONS.values()]) if DECISIONS else float("nan")
print(f"decisions: {len(DECISIONS)} in {CTX_SEC:.0f}s | judge sources: {pd.Series([d['source'] for d in DECISIONS.values()]).value_counts().to_dict()}")
print(f"rule cited is one of the 3 retrieved rules: {100 * cited_ok:.1f}%")

def final_sev(c, use_context):
    return [pl.decide(h, DECISIONS.get(f"{c}#{k}"), use_context)["severity"] for k, h in enumerate(HITS[c])]
def inn_fp(use_context):
    return sum(any(s != "none" for s in final_sev(c, use_context)) for c in INN_META)
n_inn = len(INN_META)
FP_NO, FP_CTX = inn_fp(False), inn_fp(True)
print(f"\nInnocent sentences censored WITHOUT context check: {fmt_ci(FP_NO, n_inn)}")
print(f"Innocent sentences censored WITH context check:    {fmt_ci(FP_CTX, n_inn)}")

def final_score(clips, use_context):
    tp = fp = fn = 0
    for c in clips:
        m = BENCH[c]; got = False
        for h, s in zip(HITS[c], final_sev(c, use_context)):
            if s == "none":
                continue
            if m["pos"] and h.end >= m["gt"][0] - TOL and h.start <= m["gt"][1] + TOL:
                got = True
            else:
                fp += 1
        if m["pos"]:
            tp += got; fn += not got
    return tp, fp, fn
FINAL = {}
for uc in (False, True):
    tp, fp, fn = final_score(TEST, uc); FINAL[uc] = (tp, fp, fn)
    print(f"TEST after decision ({'with' if uc else 'without'} context): precision {fmt_ci(tp, tp + fp)} | recall {fmt_ci(tp, tp + fn)}")
lost = [c for c in TEST if BENCH[c]["pos"] and any(
    h.end >= BENCH[c]["gt"][0] - TOL and h.start <= BENCH[c]["gt"][1] + TOL and s == "none"
    for h, s in zip(HITS[c], final_sev(c, True)))]
print("true insertions the context check wrongly cleared (TEST):", len(lost))
per_cat = pd.DataFrame([dict(cat=m["category"], lang=m["lang"], censored=any(s != "none" for s in final_sev(c, True)))
                        for c, m in INN_META.items()])
print(per_cat.groupby("lang")["censored"].agg(["sum", "count"]))
print("✅ Cell 16 OK | LLM usage:", dict(judge.usage))

⏱ **Runtime:** 30 s – 3 min (≈10–20 batched Gemini calls; 0 on re-run).

📋 **Check / paste back:** routed count, judge sources, rule-cited %, the two innocent FP lines, and TEST precision/recall with/without context.

## Step 8 — Bleep boundaries: (a) raw Whisper, (b) fixed padding, (c) energy-snapped
*Hypothesis before running:* Whisper word ends are often early/late by 50–150 ms, so (a) leaks; (b) fixes leak by muting clean audio;
(c) should get (b)'s leak with less over-mute. Both (b) and (c) are tuned on **dev** with the same cost
`leak% + over-mute_ms/10` (10 ms of muted clean audio = 1 % of leaked word), then reported on **test**, on true-positive hits only.

In [ ]:
# Cell 17: tune (b) and (c) on dev, compare on test with paired bootstrap
import matplotlib.pyplot as plt
from safecut.audio import EnergyCurve, pad_window, refine_window, leak_overmute
from safecut.metrics import bootstrap_mean_ci
def tp_hits(clips):
    out = []
    for c in clips:
        m = BENCH[c]
        if not m["pos"]:
            continue
        hs = [h for h in HITS[c] if h.end >= m["gt"][0] - TOL and h.start <= m["gt"][1] + TOL]
        if hs:
            out.append((c, min(h.start for h in hs), max(h.end for h in hs)))
    return out
CURVES = {c: EnergyCurve(load_audio(f"{PROJ}/clips/{c}.wav")) for c in BENCH if BENCH[c]["pos"]}
def evaluate(clips, fn):
    r = [leak_overmute(BENCH[c]["gt"], fn(c, s, e)) for c, s, e in tp_hits(clips)]
    return np.array([x[0] for x in r]), np.array([x[1] for x in r])
cost = lambda lk, om: 100 * lk.mean() + om.mean() / 10
pads = np.arange(0, 0.301, 0.025)
dev_pad = [(p, *evaluate(DEV, lambda c, s, e, p=p: pad_window(s, e, p, CURVES[c].dur))) for p in pads]
PAD = min(dev_pad, key=lambda x: cost(x[1], x[2]))[0]
ref_grid = [dict(alpha=a, min_sil=ms, guard=g) for a in (0.15, 0.25, 0.35, 0.45) for ms in (0.02, 0.04) for g in (0.0, 0.02, 0.04)]
dev_ref = [(p, *evaluate(DEV, lambda c, s, e, p=p: refine_window(CURVES[c], s, e, **p))) for p in ref_grid]
REFINE = min(dev_ref, key=lambda x: cost(x[1], x[2]))[0]
print(f"dev-chosen pad = {PAD * 1000:.0f} ms | dev-chosen refine = {REFINE}")

meth = {"(a) raw Whisper": lambda c, s, e: (s, e),
        f"(b) fixed pad {PAD * 1000:.0f} ms": lambda c, s, e: pad_window(s, e, PAD, CURVES[c].dur),
        "(c) energy-snapped ±300 ms": lambda c, s, e: refine_window(CURVES[c], s, e, **REFINE)}
BOUND = {}
for name, fn in meth.items():
    lk, om = evaluate(TEST, fn); BOUND[name] = (lk, om)
n_tp = len(tp_hits(TEST))
tab = pd.DataFrame([{"method": k, "mean leak %": round(100 * v[0].mean(), 1),
                     "words with any leak (>1%)": f"{(v[0] > 0.01).sum()}/{len(v[0])}",
                     "mean over-mute ms": round(v[1].mean(), 0), "median over-mute ms": round(np.median(v[1]), 0)}
                    for k, v in BOUND.items()])
print(f"TEST, {n_tp} correctly detected words:\n" + tab.to_string(index=False))
(lb, ob), (lc, oc) = list(BOUND.values())[1], list(BOUND.values())[2]
d_leak = bootstrap_mean_ci(100 * (lc - lb)); d_om = bootstrap_mean_ci(oc - ob)
print(f"(c) − (b) leak: {d_leak[0]:+.1f} pts, 95% CI [{d_leak[1]:+.1f}, {d_leak[2]:+.1f}]")
print(f"(c) − (b) over-mute: {d_om[0]:+.0f} ms, 95% CI [{d_om[1]:+.0f}, {d_om[2]:+.0f}]")
plt.figure(figsize=(5.5, 4))
plt.plot([o.mean() for _, _, o in dev_pad], [100 * l.mean() for _, l, _ in dev_pad], "o-", label="(b) padding 0–300 ms (dev)")
plt.scatter([o.mean() for _, _, o in dev_ref], [100 * l.mean() for _, l, _ in dev_ref], c="C3", s=18, label="(c) refine grid (dev)")
plt.xlabel("mean over-mute (ms)"); plt.ylabel("mean leak (%)"); plt.legend(); plt.title("Leak vs over-mute trade-off (dev)")
plt.tight_layout(); plt.savefig(f"{PROJ}/out/boundary_tradeoff.png", dpi=130); plt.show()
RUN_CFG = dict(refine=REFINE, pad=PAD, fuzzy_threshold=FUZZY_THR, common_zipf=ZIPF, ocr_every=CFG["OCR_EVERY"], ocr_mode="forward")
json.dump(RUN_CFG, open(f"{PROJ}/run_cfg.json", "w"))
assert n_tp > 0
print("✅ Cell 17 OK")

⏱ **Runtime:** 20–60 s (72-point refine grid on dev).

📋 **Check / paste back:** the table, the two (c)−(b) lines with CIs, and the plot.

## Step 9 — On-screen text: EasyOCR every 0.5 s, Gaussian blur, hold until the next sample
The spec's *forward hold* can't blur text that appears between samples. Because highlights are processed offline, I also test a *both-sides hold* (also blur back to the previous sample).

In [ ]:
%%writefile safecut/ocr.py
"""On-screen text: OCR every `every` seconds, blur boxes whose text matches the lexicon, hold the blur."""
import cv2
import numpy as np
from .lexicon import norm_token


def read_frames(path):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frames = []
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        frames.append(fr)
    cap.release()
    return frames, fps


def sample_indices(n_frames, fps, every=0.5):
    return sorted({min(n_frames - 1, int(round(k * every * fps))) for k in range(int(n_frames / fps / every) + 1)})


def ocr_hits(reader, matcher, frame, pad=0.1):
    """Boxes (x0,y0,x1,y1) whose OCR text contains a listed word, plus their tiers."""
    H, W = frame.shape[:2]
    out = []
    for pts, text, conf in reader.readtext(frame):
        toks = [norm_token(t) for t in text.split()]
        cands = toks + ["".join(toks)]
        m = next((r for r in (matcher.match_token(t) for t in cands if t) if r), None)
        if not m:
            continue
        xs, ys = [p[0] for p in pts], [p[1] for p in pts]
        dx, dy = pad * (max(xs) - min(xs)), pad * (max(ys) - min(ys))
        box = (max(0, min(xs) - dx), max(0, min(ys) - dy), min(W, max(xs) + dx), min(H, max(ys) + dy))
        out.append({"box": tuple(float(v) for v in box), "entry_id": m[0].id, "tier": matcher.info(m[0])[0],
                    "match": m[1], "conf": float(conf)})
    return out


def active_boxes(sample_idx, sample_boxes, k, mode="forward"):
    """Boxes applied to frame k. forward = hold each sample until the next one (live-safe).
    both = also apply to frames before the sample (possible offline, because we see the future)."""
    j = int(np.searchsorted(sample_idx, k, side="right")) - 1
    boxes = list(sample_boxes[j]) if j >= 0 else []
    if mode == "both" and j + 1 < len(sample_idx) and k != sample_idx[j]:
        boxes += sample_boxes[j + 1]
    return boxes


def blur_video(frames, fps, sample_idx, sample_boxes, out_path, mode="forward", ksize=51):
    H, W = frames[0].shape[:2]
    vw = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    for k, fr in enumerate(frames):
        fr = fr.copy()
        for x0, y0, x1, y1 in active_boxes(sample_idx, sample_boxes, k, mode):
            x0, y0, x1, y1 = int(x0), int(y0), int(np.ceil(x1)), int(np.ceil(y1))
            if x1 > x0 and y1 > y0:
                fr[y0:y1, x0:x1] = cv2.GaussianBlur(fr[y0:y1, x0:x1], (ksize, ksize), 0)
        vw.write(fr)
    vw.release()

⏱ **Runtime:** < 1 s.

📋 **Check / paste back:** the printed output

In [ ]:
# Cell 19: OCR + blur recall (frame-level, box ≥90% covered), instance recall, wrong-blur rate on clean frames
import safecut.ocr as ocr; importlib.reload(ocr)
from safecut.metrics import box_coverage
OCR_SB = {}
if CFG["SKIP_OCR"]:
    print("SKIP_OCR=True -> skipped"); OCR_RES = None
else:
    import easyocr
    reader = easyocr.Reader(["en"], gpu=True, verbose=False)
    ocr_clips = [c for c, m in BENCH.items() if m["overlays"]]
    t = time.time(); n_samples = 0
    for c in ocr_clips:
        frames, fps = ocr.read_frames(f"{PROJ}/clips/{c}.mp4")
        si = ocr.sample_indices(len(frames), fps, CFG["OCR_EVERY"])
        OCR_SB[c] = (si, [[h["box"] for h in ocr.ocr_hits(reader, MATCHER, frames[k])] for k in si], len(frames), fps)
        n_samples += len(si)
    OCR_SEC = time.time() - t
    print(f"OCR on {n_samples} sampled frames in {OCR_SEC:.0f}s ({1000 * OCR_SEC / n_samples:.0f} ms/frame)")
    OCR_RES = {}
    for mode in ("forward", "both"):
        cov_ok = n_gt = blur_clean = n_clean = inst_hit = n_inst = 0
        for c in ocr_clips:
            si, sb, nf, fps = OCR_SB[c]
            offs = [o for o in BENCH[c]["overlays"] if o["offensive"]]
            for o in offs:
                n_inst += 1
                inst_hit += any(box_coverage(o["box"], sb[j]) >= 0.9 for j, k in enumerate(si) if o["t0"] <= k / fps < o["t1"])
            for k in range(nf):
                gts = [o["box"] for o in offs if o["t0"] <= k / fps < o["t1"]]
                act = ocr.active_boxes(si, sb, k, mode)
                if gts:
                    n_gt += len(gts); cov_ok += sum(box_coverage(g, act) >= 0.9 for g in gts)
                else:
                    n_clean += 1; blur_clean += bool(act)
        OCR_RES[mode] = dict(frame_recall=(cov_ok, n_gt), instance_recall=(inst_hit, n_inst), wrong_blur=(blur_clean, n_clean))
        print(f"[{mode:7s}] frame blur recall {fmt_ci(cov_ok, n_gt)} | instance recall {fmt_ci(inst_hit, n_inst)} | frames blurred with no offensive text {fmt_ci(blur_clean, n_clean)}")
    decoy_hits = sum(bool(b) for c in ocr_clips for b in OCR_SB[c][1]
                     if not any(o["offensive"] for o in BENCH[c]["overlays"]))
    print("samples with a blur in decoy-only clips:", decoy_hits)
    assert OCR_RES["both"]["frame_recall"][0] >= OCR_RES["forward"]["frame_recall"][0]
print("✅ Cell 19 OK")

⏱ **Runtime:** 2–5 min (EasyOCR on ~300–400 sampled 720p frames).

👉 **Manual step:** If you are out of time, set SKIP_OCR=True in Cell 1 and re-run only this cell.

📋 **Check / paste back:** the two [forward]/[both] lines and the decoy line.

## Step 10 — Clip policy + render (ffmpeg) + JSON report + highlight reel
mild/strong → 1 kHz bleep over the (c) window; slur → drop the whole Whisper segment containing it. The report stores tier, action and rule — never the word.

In [ ]:
# Cell 21: render every benchmark clip, verify the bleep and drop really happened in the output files
import safecut.render as rd; importlib.reload(rd); importlib.reload(pl)
REPORT, RENDER_SEC, checks = [], 0.0, []
for c, m in BENCH.items():
    y = load_audio(f"{PROJ}/clips/{c}.wav")
    events, bleeps, drops = pl.plan_actions(c, y, ASR[c]["words"], HITS[c], DECISIONS, RUN_CFG, entries=ENTRIES)
    t = time.time()
    vid = f"{PROJ}/clips/{c}.mp4"
    if c in OCR_SB and any(OCR_SB[c][1]):
        si, sb, nf, fps = OCR_SB[c]; frames, fps = ocr.read_frames(vid)
        vid = f"{PROJ}/out/{c}_blur.mp4"; ocr.blur_video(frames, fps, si, sb, vid, mode=RUN_CFG["ocr_mode"])
        events += [{"clip": c, "modality": "video", "start": round(k / fps, 3), "end": round(k / fps + CFG["OCR_EVERY"], 3),
                    "tier": "listed", "action": "blur", "rule_id": "R13", "judge": "lexicon"} for k, b in zip(si, sb) if b]
    out = f"{PROJ}/out/{c}_safe.mp4"
    rd.render_clip(vid, f"{PROJ}/clips/{c}.wav", out, bleeps, drops)
    RENDER_SEC += time.time() - t
    REPORT += events
    if bleeps and not drops:
        yo = load_audio(out)
        for s, e in bleeps:
            seg = yo[int(s * SR) + 80:int(e * SR) - 80]
            if len(seg) > 400:
                checks.append(rd.tone_ratio(seg, SR))
    if drops:
        exp = probe_duration(f"{PROJ}/clips/{c}.wav") - sum(e - s for s, e in drops)
        assert abs(probe_duration(out) - exp) < 0.2, (c, probe_duration(out), exp)
print(f"rendered {len(BENCH)} clips in {RENDER_SEC:.0f}s | events: {pd.Series([e['action'] for e in REPORT]).value_counts().to_dict()}")
if checks:
    print(f"bleep windows checked in output audio: {len(checks)}, min 1 kHz power share = {min(checks):.3f}")
    assert min(checks) > 0.8, "a bleep window is not dominated by the 1 kHz tone"
json.dump(REPORT, open(f"{PROJ}/out/report.json", "w"), indent=1, ensure_ascii=False)
rep_txt = json.dumps(REPORT).lower()
leaks = [e.id for e in ENTRIES if len(e.text) >= 4 and re.search(r"\b" + re.escape(e.text) + r"\b", rep_txt)]
print("listed words found in report text:", len(leaks)); assert not leaks
reel = [f"{PROJ}/out/{c}_safe.mp4" for c in list(BENCH)[:60] if any(e["clip"] == c for e in REPORT)][:12]
rd.concat(reel, f"{PROJ}/out/highlight_reel.mp4")
print("reel:", len(reel), "clips,", round(probe_duration(f"{PROJ}/out/highlight_reel.mp4"), 1), "s")
print(pd.DataFrame(REPORT).head(8).drop(columns=["reason"], errors="ignore").to_string(index=False))
print("✅ Cell 21 OK")

⏱ **Runtime:** 3–6 min (100 libx264 encodes + reel).

📋 **Check / paste back:** the event counts, the bleep check line, the 'listed words found' line, and the report head.

In [ ]:
# Cell 22: before / after for one clip with an event
from IPython.display import Video, display
c = next(e["clip"] for e in REPORT if e["action"] in ("bleep", "drop"))
print(c, [ (e["action"], e["tier"], e["rule_id"]) for e in REPORT if e["clip"] == c])
display(Video(f"{PROJ}/clips/{c}.mp4", embed=True, width=420)); display(Video(f"{PROJ}/out/{c}_safe.mp4", embed=True, width=420))

⏱ **Runtime:** < 10 s.

📋 **Check / paste back:** nothing — just listen: the word should be fully covered by the tone.

## Step 11 — Results table, speed, and hypothesis check (all numbers computed here, saved to `results.json`)

In [ ]:
# Cell 24: assemble results.json (the README is generated from this file only)
total_audio = sum(BENCH[c]["dur"] for c in BENCH)
asr_sec = sum(ASR[c]["sec"] for c in BENCH)
ocr_sec = OCR_SEC * len(BENCH) / max(1, len(OCR_SB)) if OCR_SB else 0.0     # OCR ran on a subset; scale to all clips
proc = asr_sec + ocr_sec + RENDER_SEC + CTX_SEC
tp, fp, fn = score(MATCHER, TEST)[:3]
RES = dict(
    n_clips=len(BENCH), n_test=len(TEST), n_test_pos=int(sum(BENCH[c]["pos"] for c in TEST)), n_innocent=n_inn,
    detection_precision=wilson(tp, tp + fp), detection_recall=wilson(tp, tp + fn),
    exact_only_recall=wilson(tp0, tp0 + fn0),
    final_precision_ctx=wilson(FINAL[True][0], FINAL[True][0] + FINAL[True][1]), final_recall_ctx=wilson(FINAL[True][0], FINAL[True][0] + FINAL[True][2]),
    final_precision_noctx=wilson(FINAL[False][0], FINAL[False][0] + FINAL[False][1]), final_recall_noctx=wilson(FINAL[False][0], FINAL[False][0] + FINAL[False][2]),
    innocent_fp_noctx=wilson(FP_NO, n_inn), innocent_fp_ctx=wilson(FP_CTX, n_inn), rule_cited_in_top3=float(cited_ok),
    boundary={k: dict(leak_pct=float(100 * v[0].mean()), any_leak=int((v[0] > 0.01).sum()), n=len(v[0]),
                      overmute_ms=float(v[1].mean())) for k, v in BOUND.items()},
    boundary_c_minus_b_leak=d_leak, boundary_c_minus_b_overmute=d_om,
    ocr={k: {kk: wilson(*vv) for kk, vv in v.items()} for k, v in OCR_RES.items()} if OCR_RES else None,
    speed=dict(audio_s=total_audio, asr_s=asr_sec, ocr_s_scaled=ocr_sec, render_s=RENDER_SEC, llm_s=CTX_SEC,
               asr_x_rt=total_audio / asr_sec, end_to_end_x_rt=total_audio / proc,
               end_to_end_x_rt_no_llm=total_audio / (proc - CTX_SEC)),
    asr_wer_clean=float(wer), run_cfg=RUN_CFG, judge_usage=dict(judge.usage), gpu=torch.cuda.get_device_name(0),
    whisper=CFG["WHISPER"])
json.dump(RES, open(f"{PROJ}/results.json", "w"), indent=1, default=float)
p = lambda x: f"{100 * x[0]:.1f}% [{100 * x[1]:.1f}–{100 * x[2]:.1f}]"
print("Detection (test)  P", p(RES["detection_precision"]), "| R", p(RES["detection_recall"]), "| exact-only R", p(RES["exact_only_recall"]))
print("Final, no context P", p(RES["final_precision_noctx"]), "| R", p(RES["final_recall_noctx"]))
print("Final, context    P", p(RES["final_precision_ctx"]), "| R", p(RES["final_recall_ctx"]))
print("Innocent FP       no-context", p(RES["innocent_fp_noctx"]), "| context", p(RES["innocent_fp_ctx"]))
for k, v in RES["boundary"].items():
    print(f"{k:30s} leak {v['leak_pct']:.1f}% | any-leak {v['any_leak']}/{v['n']} | over-mute {v['overmute_ms']:.0f} ms")
if RES["ocr"]:
    print("OCR blur recall (frames) forward", p(RES["ocr"]["forward"]["frame_recall"]), "| both", p(RES["ocr"]["both"]["frame_recall"]))
s = RES["speed"]; print(f"Speed on {RES['gpu']}: ASR {s['asr_x_rt']:.1f}x RT | end-to-end {s['end_to_end_x_rt']:.1f}x RT ({s['end_to_end_x_rt_no_llm']:.1f}x without LLM time)")
print("\nHypothesis checks:")
print("  context check lowers innocent FP:", FP_CTX < FP_NO)
print("  context check costs recall on test:", FINAL[True][0] < FINAL[False][0])
print("  fuzzy/translit adds recall over exact-only:", tp > tp0)
print("  (c) leaks less than (b):", RES["boundary_c_minus_b_leak"][0] < 0, "| (c) over-mutes less than (b):", RES["boundary_c_minus_b_overmute"][0] < 0)
print("✅ Cell 24 OK")

⏱ **Runtime:** < 5 s.

📋 **Check / paste back:** this whole printout — the README and resume bullets are built from it.

## Step 12 — Streamlit app (upload → before/after, report table, rules cited)

In [ ]:
%%writefile app.py
import json, os, sys, tempfile, torch
import pandas as pd, streamlit as st
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from safecut import lexicon as lx, llm, rag, pipeline as pl
PROJ = os.environ.get("SAFECUT_PROJ", "/kaggle/working/safecut_run")

@st.cache_resource
def load_ctx():
    import easyocr
    from faster_whisper import WhisperModel
    cfg = json.load(open(f"{PROJ}/run_cfg.json"))
    entries = lx.load_entries(lx.download_lists(f"{PROJ}/lists"))
    tiers = {int(k): v for k, v in json.load(open(f"{PROJ}/tiers.json")).items()}
    policy = next(p for p in [os.environ.get("SAFECUT_POLICY", ""), f"{PROJ}/content_policy.md"] if p and os.path.exists(p))
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    return dict(cfg=cfg, entries=entries, matcher=lx.Matcher(entries, tiers, cfg["fuzzy_threshold"], cfg["common_zipf"]),
                judge=llm.Judge(f"{PROJ}/cache/llm_cache.json", api_key=os.environ.get("GEMINI_API_KEY"), local_device=dev),
                index=rag.PolicyIndex(rag.load_policy(policy), device=dev),
                whisper=WhisperModel(os.environ.get("SAFECUT_WHISPER", "large-v3"), device=dev, compute_type="float16" if dev == "cuda" else "int8"),
                reader=easyocr.Reader(["en"], gpu=dev == "cuda", verbose=False), rules={r["rule_id"]: r for r in rag.load_policy(policy)})

st.set_page_config(page_title="SafeCut", layout="wide")
st.title("SafeCut — context-aware censoring for sports highlights")
up = st.file_uploader("Upload a short clip (mp4)", type=["mp4", "mov", "mkv"])
if up:
    ctx = load_ctx()
    work = tempfile.mkdtemp(); src = os.path.join(work, "upload.mp4"); open(src, "wb").write(up.read())
    with st.spinner("ASR → detection → context check → OCR blur → render ..."):
        report, out = pl.process_video(src, ctx, work)
    a, b = st.columns(2)
    a.subheader("Before"); a.video(src)
    b.subheader("After"); b.video(str(out))
    ev = pd.DataFrame(report["events"])
    st.subheader("Report (tier and action only — the word is never shown)")
    st.dataframe(ev.drop(columns=["heard_masked"], errors="ignore") if len(ev) else pd.DataFrame([{"result": "nothing censored"}]))
    st.json(report["timings_s"])
    st.subheader("Rules cited")
    for rid in sorted(set(ev["rule_id"])) if len(ev) else []:
        r = ctx["rules"].get(rid)
        if r:
            st.markdown(f"**{rid} — {r['title']}**: {r['text']}")

⏱ **Runtime:** < 1 s.

📋 **Check / paste back:** the printed output

In [ ]:
# Cell 26 (optional): run the app on Kaggle through a temporary Cloudflare tunnel
shutil.copy(POLICY, f"{PROJ}/content_policy.md")
if not os.path.exists("/kaggle/working/cloudflared"):
    subprocess.run(["wget", "-q", "-O", "/kaggle/working/cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod("/kaggle/working/cloudflared", 0o755)
env = dict(os.environ, SAFECUT_PROJ=PROJ, GEMINI_API_KEY=GEMINI_KEY or "")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"], env=env,
                 stdout=open("/tmp/st.log", "w"), stderr=subprocess.STDOUT)
tun = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://localhost:8501"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tun.stdout:                       # first lines are status messages; wait for the real URL
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        print("Open:", m.group(0)); break
print("Upload e.g.", f"{PROJ}/clips/{c}.mp4", "(download it from the Output panel first)")

⏱ **Runtime:** 20–40 s to get the link; the first upload takes ~1 min (models load).

👉 **Manual step:** Click the printed trycloudflare link, upload a clip, take a screenshot of before/after + report for the README. The link dies when the session stops.

📋 **Check / paste back:** the link line (or skip this cell if short on time).

## Step 13 — README + resume bullets, generated only from `results.json`

In [ ]:
# Cell 28: write README.md from results.json (no number is typed by hand)
R = json.load(open(f"{PROJ}/results.json")); q = lambda x: f"{100 * x[0]:.1f}% (95% CI {100 * x[1]:.1f}–{100 * x[2]:.1f})"
bd = R["boundary"]; names = list(bd)
rows = "\n".join(f"| {k} | {v['leak_pct']:.1f}% | {v['any_leak']}/{v['n']} | {v['overmute_ms']:.0f} ms |" for k, v in bd.items())
ocr_rows = "" if not R["ocr"] else "\n".join(
    f"| OCR blur recall, frames ({m} hold) | {q(v['frame_recall'])} |\n| Frames blurred without offensive text ({m}) | {q(v['wrong_blur'])} |"
    for m, v in R["ocr"].items())
readme = f"""# SafeCut — context-aware profanity & slur censoring for sports highlights (English + Hinglish)

All numbers come from `results.json`, produced by the Kaggle notebook ({R['gpu']}, Whisper {R['whisper']}).

## Pipeline
```mermaid
flowchart LR
  V[clip] --> A[ffmpeg 16 kHz audio] --> W[faster-whisper large-v3<br/>word timestamps, auto language]
  W --> D[detect: exact / phrase / fuzzy / Hinglish skeleton / Devanagari translit]
  D -->|ambiguous| C[RAG: FAISS top-3 policy rules -> Gemini Flash JSON]
  D -->|unambiguous| P
  C --> P[policy: mild/strong = bleep, slur = drop segment]
  W --> B[bleep window snapped to low-energy points ±300 ms]
  V --> O[EasyOCR every {R['run_cfg']['ocr_every']} s -> Gaussian blur, held]
  B --> F[ffmpeg: 1 kHz bleep + drop + concat] 
  P --> F
  O --> F --> OUT[safe clip + reel + JSON report]
```

## Benchmark (synthetic, exact ground truth)
{R['n_clips']} clean MELD test clips; half get one listed word (TTS) spliced at a pause with video frozen for the same time;
20 get offensive on-screen text, 10 get clean decoy text; {R['n_innocent']} innocent sentences where a listed word is used innocently.
Thresholds tuned on a 30% dev split; numbers below are on the {R['n_test']}-clip test split ({R['n_test_pos']} inserted words).

## Results
| Metric | Value |
|---|---|
| Detection precision (test) | {q(R['detection_precision'])} |
| Detection recall (test) | {q(R['detection_recall'])} |
| Recall with exact matching only | {q(R['exact_only_recall'])} |
| Final precision / recall, no context check | {q(R['final_precision_noctx'])} / {q(R['final_recall_noctx'])} |
| Final precision / recall, with context check | {q(R['final_precision_ctx'])} / {q(R['final_recall_ctx'])} |
| Innocent sentences censored, no context check | {q(R['innocent_fp_noctx'])} |
| Innocent sentences censored, with context check | {q(R['innocent_fp_ctx'])} |
| Rule cited is in retrieved top-3 | {100 * R['rule_cited_in_top3']:.1f}% |
{ocr_rows}
| ASR speed | {R['speed']['asr_x_rt']:.1f}× real-time |
| End-to-end speed (incl. LLM calls) | {R['speed']['end_to_end_x_rt']:.1f}× real-time |
| Whisper WER on untouched clips | {R['asr_wer_clean']:.3f} |

### Bleep boundaries (test, correctly detected words)
| Window | Mean leak | Words with any leak | Mean over-mute |
|---|---|---|---|
{rows}

(c) − (b): leak {R['boundary_c_minus_b_leak'][0]:+.1f} pts (95% CI {R['boundary_c_minus_b_leak'][1]:+.1f} to {R['boundary_c_minus_b_leak'][2]:+.1f}),
over-mute {R['boundary_c_minus_b_overmute'][0]:+.0f} ms (95% CI {R['boundary_c_minus_b_overmute'][1]:+.0f} to {R['boundary_c_minus_b_overmute'][2]:+.0f}).

![trade-off](boundary_tradeoff.png)

## Limitations
- Synthetic benchmark: TTS voices differ from live crowd / commentator audio; the inserted word does not fit the sentence, and pauses around it make boundaries easier than real speech.
- Small n (confidence intervals are wide); one random seed.
- Tiers come from an LLM pass over the LDNOOBW lists (cached in `tiers.json`), not from human annotators.
- OCR is English-only (EasyOCR `en`); Devanagari on-screen text is not handled. OCR every 0.5 s can miss text that appears briefly.
- LLM used: {R['judge_usage']} (cache hits count as `cache`).
"""
open(f"{W}/README.md", "w").write(readme); shutil.copy(f"{PROJ}/out/boundary_tradeoff.png", f"{W}/boundary_tradeoff.png")
print(readme)
b = R["boundary"]; bk, ck = names[1], names[2]
print("\nRESUME BULLETS (numbers from results.json; delete any claim the table does not support):")
print(f"• Built SafeCut, a context-aware profanity/slur censoring pipeline for English+Hinglish sports clips (faster-whisper word timestamps, "
      f"fuzzy + transliteration matching, FAISS-RAG policy check with Gemini); on a {R['n_clips']}-clip synthetic benchmark with exact ground truth it reached "
      f"{100 * R['detection_recall'][0]:.0f}% word recall at {100 * R['detection_precision'][0]:.0f}% precision and cut false censoring of innocent uses "
      f"from {100 * R['innocent_fp_noctx'][0]:.0f}% to {100 * R['innocent_fp_ctx'][0]:.0f}%.")
print(f"• Designed energy-based bleep-boundary refinement: word leak {b[ck]['leak_pct']:.1f}% vs {b[names[0]]['leak_pct']:.1f}% for raw Whisper timestamps, "
      f"with {b[ck]['overmute_ms']:.0f} ms vs {b[bk]['overmute_ms']:.0f} ms over-mute compared to fixed padding; full pipeline runs at "
      f"{R['speed']['end_to_end_x_rt']:.1f}× real-time on a {R['gpu']}.")

⏱ **Runtime:** < 5 s.

👉 **Manual step:** Copy `README.md`, `boundary_tradeoff.png`, `app.py`, `safecut/` from the Output panel to your GitHub repo. Read the bullets — keep only claims the table supports (e.g. if (c) did not beat (b), say so honestly). Then click **Save Version → Save & Run All** so outputs are kept.

📋 **Check / paste back:** the two bullets.